<!-- # C3 回路による量子乱数生成 -->

# Quantum Random Number Generation with the C3 Circuit

<!-- ## このノートブックで行うこと -->

## What this notebook does

<!-- 5 個の量子ビットを **GHZ 状態**というもつれ合った状態にしてから測定する **C3 回路**を Qiskit で実装し、得られる乱数の偏りとエントロピーを調べます。1 回の測定（1 shot）で 5 ビットの結果が得られますが、理想的には 5 ビットはすべて同じ値なので、**多数決**で 1 ビットの乱数にします。流れは次のとおりです。 -->

We implement the **C3 circuit**, which puts 5 qubits into an entangled state called the **GHZ state** before measuring them, in Qiskit and examine the bias and entropy of the resulting random numbers. Each measurement (1 shot) yields a 5-bit outcome, but ideally all 5 bits have the same value, so they are reduced to 1 random bit by **majority vote**. The workflow is as follows.

<!--
1. C3 回路を作る（中心の量子ビットに重ね合わせを作るゲートを Hadamard ゲート $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ の中から 1 つ選び、CX ゲートで GHZ 状態にする）
2. 回路を量子コンピュータのネイティブゲートに変換（transpile）し、CX ゲートが $\mathrm{CZ}$ ゲートに分解されることを確かめる
3. **ノイズなし**のシミュレータで測定し、多数決で 1 ビットにして評価する
4. 実機相当の**ノイズあり**のシミュレータで測定し、誤りが多数決でどの程度訂正されるかを評価する
5. 同じ測定を 10 回繰り返し、結果のばらつきを見る
6. ノイズの強さを段階的に変え、結果への影響を見る（5.9）
-->

1. Build the C3 circuit (choose one gate to create the superposition on the central qubit from the Hadamard gate $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, and $R_Y(\pi/2)$, then create the GHZ state with CX gates)
2. Convert (transpile) the circuit into the native gates of the quantum computer and confirm that the CX gates are decomposed into $\mathrm{CZ}$ gates
3. Measure on a **noiseless** simulator, reduce to 1 bit by majority vote, and evaluate
4. Measure on a **noisy** simulator that mimics real hardware and evaluate how well errors are corrected by the majority vote
5. Repeat the same measurement 10 times to see how much the results vary
6. Change the noise strength step by step to see its effect on the results (5.9)

<!-- このノートブックは、C1 回路のノートブック（`C1_circuit_textbook.ipynb`）を先に読んでいることを前提にしています。C1 と共通の内容（1 量子ビットゲートの行列、ノイズの式、$\lambda$ の決め方など）は要点だけを書くので、詳しくは C1 のノートブックの説明パートを参照してください。5 量子ビットの結果の扱い方（Qiskit のビットの並び順など）は C2 回路のノートブック（`C2_circuit_textbook.ipynb`）と共通です。 -->

This notebook assumes that you have already read the C1 circuit notebook (`C1_circuit_textbook.ipynb`). Material shared with C1 (1-qubit gate matrices, noise formulas, how $\lambda$ is chosen, and so on) is only summarized here, so see the explanatory part of the C1 notebook for details. The handling of 5-qubit outcomes (Qiskit's bit ordering and so on) is shared with the C2 circuit notebook (`C2_circuit_textbook.ipynb`).

<!-- この説明パート（A〜G）で理論と使う道具をひととおり説明し、後半の各節（1.〜5.）でその内容をコードと並べて一つずつ確かめていきます。 -->

This explanatory part (A–G) goes through the theory and the tools used, and the later sections (1.–5.) then check that material one step at a time alongside the code.

---

<!-- ### A. 先行研究と C3 回路 -->

### A. Prior work and the C3 circuit

<!-- このノートブックの回路と、ノイズのパラメータの数値は、次の論文にもとづいています。ノイズモデルの組み立て方は論文のものではなく、このノートブックで選んだものです（E を参照）。 -->

The circuit and the numerical noise parameters in this notebook are based on the following paper. How the noise model is assembled is not taken from the paper; it was chosen for this notebook (see E).

> A. Gnatowski *et al.*, "True Random Number Generators on IQM Spark," arXiv:2512.09862 (2025).

<!-- C3 回路は論文で次のように説明されています。 -->

The paper describes the C3 circuit as follows.

> "The next circuit C3 is based on the idea of maximally entangled multi-qubit state called the Greenberger–Horne–Zeilinger (GHZ) state (for 2 qubit this is called Bell's state instead). For 5 qubits this means that there is 50% chance of observing state $|00000\rangle$ and 50% chance of observing state $|11111\rangle$:"
>
> $$|\Psi_5\rangle = \frac{|00000\rangle + |11111\rangle}{\sqrt{2}} \qquad (3)$$

<!-- GHZ 状態の作り方は次のとおりです。 -->

The GHZ state is created as follows.

> "The typical way to achieve this state is to create a superposition on one qubit (using gates like H, Rx etc.) and then apply a CX (a controlled not, CNOT) gate multiple times such that the qubit in superposition is always the control qubit."

<!-- 論文の Fig. 4 と同じく、中心の量子ビット 2 に重ね合わせを作り、量子ビット 2 を制御ビットとして、量子ビット 1, 3, 0, 4 の順に CX ゲートをかけます。 -->

As in Fig. 4 of the paper, we create the superposition on the central qubit 2 and, with qubit 2 as the control, apply CX gates to qubits 1, 3, 0, and 4 in that order.

```
q0: |0⟩ ────────────────────────────X───────────[M]─
                                    │
q1: |0⟩ ────────────X───────────────┼───────────[M]─
                    │               │
q2: |0⟩ ──[ * ]─────●───────●───────●───────●───[M]─
                            │               │
q3: |0⟩ ────────────────────X───────────────┼───[M]─
                                            │
q4: |0⟩ ────────────────────────────────────X───[M]─
```

<!-- （`*` は $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ のいずれか。`●` が制御ビット、`X` が標的ビット） -->

(`*` is one of $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$. `●` is the control and `X` is the target.)

<!-- C1 のノートブックと同じく、Shannon エントロピー $H_{\mathrm{Shannon}}$ と区別するため、Hadamard ゲートを $H_{\mathrm{Gate}}$ と書きます。コード内のゲート名やグラフの凡例は `"H"` のままです。 -->

As in the C1 notebook, the Hadamard gate is written $H_{\mathrm{Gate}}$ to distinguish it from the Shannon entropy $H_{\mathrm{Shannon}}$. Gate names in the code and in plot legends remain `"H"`.

<!--
論文が挙げる C3 回路の特徴は次のとおりです。
- C1 と C2 の性質を併せ持つ。複数の量子ビットを使うが、測定結果は（理想的には）$|00000\rangle$ と $|11111\rangle$ の 2 通りだけなので、本質的には 1 個の量子ビットのように振る舞う。
- **短所**：複数の量子ビットを使うのに、生成速度は C1 と同じ（1 shot で 1 ビット）。
- **長所**：読み出し誤差をある程度訂正できる。たとえば $|01000\rangle$ が観測されたら、これは誤りの結果である。「量子ビット 1 が誤って 1 と読まれた（正しくは $|00000\rangle$）」か「量子ビット 0, 2, 3, 4 が誤って 0 と読まれた（正しくは $|11111\rangle$）」のどちらかで、前者の方がずっと起こりやすいので $|00000\rangle$ とみなせる。
- **制約**：量子ビットのつながり方（接続グラフ）に左右される。IQM Spark では量子ビット 2 が他のすべての量子ビットとつながった星形なので、量子ビット 2 は他のどの量子ビットとも直接もつれさせられるが、中心以外の 2 個の量子ビットどうしは直接もつれさせられない。
-->

The paper lists the following characteristics of the C3 circuit.
- It combines properties of C1 and C2. It uses multiple qubits, but the measurement outcome is (ideally) only $|00000\rangle$ or $|11111\rangle$, so it essentially behaves like a single qubit.
- **Disadvantage**: Although it uses multiple qubits, its generation rate is the same as C1 (1 bit per shot).
- **Advantage**: Readout errors can be corrected to some extent. For example, if $|01000\rangle$ is observed, it is an erroneous result. Either "qubit 1 was erroneously read as 1 (the correct result being $|00000\rangle$)" or "qubits 0, 2, 3, and 4 were erroneously read as 0 (the correct result being $|11111\rangle$)," and since the former is far more likely, it can be regarded as $|00000\rangle$.
- **Constraint**: It depends on how the qubits are connected (the connectivity graph). IQM Spark has a star topology in which qubit 2 is connected to all the other qubits, so qubit 2 can be directly entangled with any other qubit, but two non-central qubits cannot be directly entangled with each other.

<!-- > 例の $|01000\rangle$ は、論文では "either qubit 1 was erroneously measured as 1" と説明されています。論文の表記では左端が量子ビット 0 だと考えられます。このノートブックで使う Qiskit の文字列では右端が量子ビット 0 なので、同じ文字列 `'01000'` は量子ビット 3 が 1 であることを表します（説明パート B）。どちらの読み方でも、多数決の結果は 0 です。 -->

> The example $|01000\rangle$ is explained in the paper as "either qubit 1 was erroneously measured as 1." In the paper's notation, the leftmost character appears to be qubit 0. In the Qiskit strings used in this notebook, the rightmost character is qubit 0, so the same string `'01000'` means that qubit 3 is 1 (Explanation Part B). Under either reading, the result of the majority vote is 0.

<!-- **論文の実機の結果**（Table IV）：論文は Odra 5 で、2〜5 個の量子ビットのあらゆる組み合わせの C3 回路を試しています。5 個すべてを使った C3 回路では 1 の割合が 50.32%〜50.42% と 50% を少し上回りました。測定誤差のモデルから予想される値（"Ex"）は 49.99% です。また、2 量子ビットでは 50% より低く、3 量子ビット以上では高いという、量子ビット数の偶奇で偏りの向きが変わる現象が見られ、論文は「測定誤差だけでは説明できず、crosstalk などの影響が考えられる」と結論しています。 -->

**Results on real hardware in the paper** (Table IV): The paper tries C3 circuits on Odra 5 for every combination of 2 to 5 qubits. For the C3 circuit using all 5 qubits, the fraction of 1s was 50.32%–50.42%, slightly above 50%. The value predicted from the measurement error model ("Ex") is 49.99%. A phenomenon was also observed in which the direction of the bias changes with the parity of the number of qubits (below 50% for 2 qubits, above for 3 or more), and the paper concludes that "it cannot be explained by measurement errors alone, and effects such as crosstalk are conceivable."

---

<!-- ### B. 量子乱数生成の原理 -->

### B. Principle of quantum random number generation

<!-- **ステップ 1：重ね合わせを作る**　量子ビット 2 にゲートを作用させると、5 量子ビットの状態は（$H_{\mathrm{Gate}}$, $R_Y(\pi/2)$ の場合） -->

**Step 1: Create the superposition.** Applying a gate to qubit 2 gives the 5-qubit state (for $H_{\mathrm{Gate}}$ and $R_Y(\pi/2)$)

<!--
$$
|\psi_1\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}\ (\text{量子ビット } 2) \ \otimes\ |0\rangle|0\rangle|0\rangle|0\rangle\ (\text{量子ビット } 0, 1, 3, 4)
$$
-->

$$
|\psi_1\rangle = \frac{|0\rangle + |1\rangle}{\sqrt{2}}\ (\text{qubit } 2) \ \otimes\ |0\rangle|0\rangle|0\rangle|0\rangle\ (\text{qubits } 0, 1, 3, 4)
$$

<!-- になります。$R_X(\pi/2)$ の場合は $|1\rangle$ の係数が $-i$ になります。 -->

For $R_X(\pi/2)$, the coefficient of $|1\rangle$ becomes $-i$.

<!-- **ステップ 2：CX ゲートで値をコピーする**　CX ゲート（CNOT）は、制御ビット $c$ が 1 のときだけ標的ビット $t$ を反転します。 -->

**Step 2: Copy the value with CX gates.** The CX gate (CNOT) flips the target bit $t$ only when the control bit $c$ is 1.

$$
\mathrm{CX}\,|c\rangle|t\rangle = |c\rangle|t \oplus c\rangle, \qquad
\mathrm{CX} = \begin{pmatrix} 1 & 0 & 0 & 0 \\ 0 & 1 & 0 & 0 \\ 0 & 0 & 0 & 1 \\ 0 & 0 & 1 & 0 \end{pmatrix}
$$

<!-- （行列は基底 $|c\,t\rangle = |00\rangle, |01\rangle, |10\rangle, |11\rangle$ の順）。標的ビットはどれも $|0\rangle$ から始まるので、CX ゲートは制御ビットの値を標的ビットに「コピー」します。量子ビット 2 が重ね合わせなので、4 個の CX ゲートの後の状態は -->

(The matrix is in the order of the basis $|c\,t\rangle = |00\rangle, |01\rangle, |10\rangle, |11\rangle$.) Since every target starts in $|0\rangle$, the CX gates "copy" the value of the control to the targets. Because qubit 2 is in a superposition, the state after the 4 CX gates is

<!--
$$
|\Psi_5\rangle = \frac{|00000\rangle + |11111\rangle}{\sqrt{2}}
\qquad \left(R_X(\pi/2) \text{ の場合は } \frac{|00000\rangle - i\,|11111\rangle}{\sqrt{2}}\right)
$$
-->

$$
|\Psi_5\rangle = \frac{|00000\rangle + |11111\rangle}{\sqrt{2}}
\qquad \left(\text{for } R_X(\pi/2)\text{: } \frac{|00000\rangle - i\,|11111\rangle}{\sqrt{2}}\right)
$$

<!-- となり、論文の式 (3) の GHZ 状態になります。ボルンの規則により $P(00000) = P(11111) = 1/2$ で、それ以外の 30 通りの確率は 0 です。 -->

which is the GHZ state of Eq. (3) in the paper. By the Born rule, $P(00000) = P(11111) = 1/2$, and the probabilities of the other 30 outcomes are 0.

<!-- 各量子ビットを 1 個ずつ見ると、どれも 0 と 1 が確率 1/2 で出ます。しかし 5 個の値は常にそろっているので、1 shot から得られる乱数は 1 ビットだけです。乱数の元は量子ビット 2 の重ね合わせ 1 個で、C1 回路と同じです。もつれは、その結果を 5 個の量子ビットにコピーする役割を果たします。 -->

Looking at each qubit individually, every one gives 0 and 1 with probability 1/2. However, the 5 values always agree, so a shot yields only 1 random bit. The source of randomness is the single superposition on qubit 2, the same as in the C1 circuit. The entanglement serves to copy that result onto the 5 qubits.

<!-- **多数決による復号**　測定結果 $x$（5 ビット）の 1 の個数（ハミング重み）を $|x|$ として、 -->

**Decoding by majority vote.** With $|x|$ the number of 1s (Hamming weight) in the 5-bit measurement outcome $x$, we convert to 1 bit by

$$
d(x) = \begin{cases} 1 & (|x| \ge 3) \\ 0 & (|x| \le 2) \end{cases}
$$

<!-- で 1 ビットに変換します。量子ビットが 5 個（奇数）なので、引き分けは起こりません。理想的には $|x|$ は 0 か 5 なので、$d(x)$ は 1 個の量子ビットを測ったのと同じ公平な 1 ビットです。 -->

Since there are 5 qubits (an odd number), ties do not occur. Ideally $|x|$ is 0 or 5, so $d(x)$ is a fair bit equivalent to measuring a single qubit.

<!-- 多数決で訂正できるのは、各量子ビットで独立に起きた誤り（読み出し誤差や、標的ビットで起きたゲートの誤りなど）です。CX ゲートでコピーする前に量子ビット 2 で起きた誤りは、その後の CX ゲートで他の量子ビットにもコピーされるので、多数決では訂正できません（5.9 の熱緩和の結果を参照）。 -->

What a majority vote can correct is errors that occur independently on each qubit (readout errors, gate errors on target qubits, and so on). Errors on qubit 2 that occur before copying with the CX gates are copied to the other qubits by the subsequent CX gates, so they cannot be corrected by majority vote (see the thermal relaxation results in 5.9).

<!-- **Qiskit のビットの並び順**：C2 と同じく、Qiskit の測定結果の文字列は**右端が量子ビット 0**、左端が量子ビット 4 です。多数決は 1 の個数だけで決まるので、並び順には左右されません。 -->

**Qiskit's bit ordering**: As in C2, in Qiskit's measurement-result strings the **rightmost character is qubit 0** and the leftmost is qubit 4. The majority vote depends only on the number of 1s, so it is not affected by the ordering.

---

<!-- ### C. 評価指標 -->

### C. Evaluation metrics

<!-- $N$ を shot 数とします。C3 回路では 1 shot で 1 ビットの乱数が得られるので、乱数のビット数も $N$ です。 -->

Let $N$ be the number of shots. Since the C3 circuit yields 1 random bit per shot, the number of random bits is also $N$.

<!-- **(1) 多数決で得たビット列**：C1 と同じ指標を計算します。多数決の結果が 1 になった回数を $n_1$ として -->

**(1) Bit string from the majority vote**: We compute the same metrics as for C1. With $n_1$ the number of times the majority vote gave 1,

$$
\hat{P}(1) = \frac{n_1}{N}, \quad \mathrm{bias} = \left|\hat{P}(1) - \frac{1}{2}\right|, \quad
H_{\mathrm{Shannon}} = -\sum_{b \in \{0,1\}} \hat{P}(b) \log_2 \hat{P}(b), \quad
H_{\min} = -\log_2 \max\bigl(\hat{P}(0), \hat{P}(1)\bigr)
$$

<!-- です。統計的な揺らぎの目安は C1 と同じく $\mathbb{E}[\mathrm{bias}] \approx \sqrt{2/\pi}\,/\,(2\sqrt{N})$ です。 -->

As in C1, the reference value for statistical fluctuation is $\mathbb{E}[\mathrm{bias}] \approx \sqrt{2/\pi}\,/\,(2\sqrt{N})$.

<!-- **(2) GHZ 状態からのずれ**：$|00000\rangle$, $|11111\rangle$ 以外の結果が出た割合（non-GHZ fraction） -->

**(2) Deviation from the GHZ state**: We compute the fraction of outcomes other than $|00000\rangle$ and $|11111\rangle$ (the non-GHZ fraction)

$$
f_{\text{non-GHZ}} = 1 - \frac{n_{00000} + n_{11111}}{N}
$$

<!-- を求めます。理想的には 0 で、誤りが起きた shot の割合の目安になります。$|x| = 1, 2$ の結果は多数決で 0 に、$|x| = 3, 4$ の結果は 1 に訂正されます。どの $|x|$ が何回出たか（ハミング重みの分布）も表にします。 -->

Ideally this is 0, and it serves as a guide to the fraction of shots in which errors occurred. Outcomes with $|x| = 1, 2$ are corrected to 0 by the majority vote, and those with $|x| = 3, 4$ to 1. We also tabulate how many times each $|x|$ occurred (the Hamming-weight distribution).

<!-- **(3) 5 ビットの結果の min-entropy**：32 通りの結果の分布の min-entropy $H_{\min}^{\mathrm{raw}} = -\log_2 \max_x \hat{P}(x)$ です。理想的には $\max_x \hat{P}(x) = 1/2$ なので **1 ビット**で、5 ビットの結果にも乱数は 1 ビット分しか含まれないことを表します。ノイズがあると 1 より大きくなりますが、増えた分は誤りによるもので、乱数の元として当てにできるものではありません。 -->

**(3) Min-entropy of the 5-bit outcomes**: the min-entropy $H_{\min}^{\mathrm{raw}} = -\log_2 \max_x \hat{P}(x)$ of the distribution of the 32 outcomes. Ideally $\max_x \hat{P}(x) = 1/2$, so it is **1 bit**, showing that even the 5-bit outcome contains only 1 bit of randomness. With noise it becomes larger than 1, but the increase is due to errors and cannot be relied on as a source of randomness.

<!-- **(4) （参考）事後選択**：多数決の代わりに、$|00000\rangle$ と $|11111\rangle$ の shot だけを使い、それ以外を捨てる方法もあります。このときの 1 の割合は $\hat{P}_{\mathrm{ps}}(1) = n_{11111} / (n_{00000} + n_{11111})$ です。捨てた分だけ生成速度は下がります。 -->

**(4) (Optional) Postselection**: Instead of a majority vote, one can use only the shots giving $|00000\rangle$ or $|11111\rangle$ and discard the rest. The fraction of 1s is then $\hat{P}_{\mathrm{ps}}(1) = n_{11111} / (n_{00000} + n_{11111})$. The generation rate decreases by the amount discarded.

<!-- **読み出し誤差だけのときの理論値**：理想的な GHZ 状態を、各量子ビットが独立に確率 $\varepsilon$ で読み間違える（対称な読み出し誤差）とします。正しい結果がそのまま読まれる確率は $(1-\varepsilon)^5$、5 個すべてが反転してもう一方の GHZ の結果になる確率は $\varepsilon^5$ なので -->

**Theoretical values with readout error only**: Suppose each qubit of an ideal GHZ state is independently misread with probability $\varepsilon$ (symmetric readout error). The probability that the correct result is read as is equals $(1-\varepsilon)^5$, and the probability that all 5 flip to give the other GHZ outcome equals $\varepsilon^5$, so

$$
f_{\text{non-GHZ}} = 1 - (1-\varepsilon)^5 - \varepsilon^5 \approx 0.141 \quad (\varepsilon = 0.03)
$$

<!-- です。多数決が誤るのは 3 個以上が反転したときなので、その確率は -->

The majority vote fails when 3 or more bits flip, so its probability is

$$
M(\varepsilon) = \sum_{k=3}^{5} \binom{5}{k} \varepsilon^k (1-\varepsilon)^{5-k} \approx 2.6 \times 10^{-4} \quad (\varepsilon = 0.03)
$$

<!-- で、1 個の量子ビットの誤り確率 $\varepsilon = 0.03$ より 2 桁小さくなります。 -->

which is two orders of magnitude smaller than the single-qubit error probability $\varepsilon = 0.03$.

<!-- **多数決が偏りを抑えるしくみ**：元のビットは公平（$P(1) = 1/2$）なので、0→1 と 1→0 が同じ確率で起こる対称な誤りは、多数決をしなくても偏りを生みません。偏りの原因になるのは非対称な読み出し誤差 $P(1|0) = a$, $P(0|1) = b$ です。論文は、C1 回路の 1 の割合を -->

**How the majority vote suppresses bias**: The original bit is fair ($P(1) = 1/2$), so symmetric errors, in which 0→1 and 1→0 occur with equal probability, produce no bias even without a majority vote. Bias arises from asymmetric readout error $P(1|0) = a$, $P(0|1) = b$. The paper writes the fraction of 1s for the C1 circuit as

$$
p_{\mathrm{obs}} = \frac{1}{2}\bigl(1 - P(0|1)\bigr) + \frac{1}{2} P(1|0) = \frac{1}{2} - \frac{P(0|1) - P(1|0)}{2}
$$

<!-- と書いています。C3 回路で多数決をすると、1 個の量子ビットの誤り確率 $a$, $b$ が多数決の誤り確率 $M(a)$, $M(b)$ に置き換わるので -->

With the majority vote in the C3 circuit, the single-qubit error probabilities $a$ and $b$ are replaced by the majority-vote error probabilities $M(a)$ and $M(b)$, so

$$
p_{\mathrm{obs}}^{\mathrm{C3}} = \frac{1}{2} - \frac{M(b) - M(a)}{2}
$$

<!-- です。たとえば $a = 0$, $b = 0.02$ なら、C1 の bias は $0.01$、C3 の bias は $M(0.02)/2 \approx 3.9 \times 10^{-5}$ です。$M(\delta) \approx 10\,\delta^3$ なので、偏りは誤り確率の 3 乗のオーダーまで抑えられます。5.9 でこれを確かめます。 -->

For example, with $a = 0$ and $b = 0.02$, the bias of C1 is $0.01$ and the bias of C3 is $M(0.02)/2 \approx 3.9 \times 10^{-5}$. Since $M(\delta) \approx 10\,\delta^3$, the bias is suppressed to the order of the cube of the error probability. This is verified in 5.9.

---

<!-- ### D. ネイティブゲートと transpile -->

### D. Native gates and transpilation

<!-- IQM Spark（Odra 5）のネイティブゲートは $R_X$, $R_Y$（1 量子ビット）と $\mathrm{CZ}$（2 量子ビット）です（論文の Table II）。CX ゲートはネイティブゲートではないので、transpile で -->

The native gates of IQM Spark (Odra 5) are $R_X$, $R_Y$ (1-qubit) and $\mathrm{CZ}$ (2-qubit) (Table II of the paper). The CX gate is not a native gate, so transpilation decomposes it as

$$
\mathrm{CX}_{c \to t} = (I \otimes H_{\mathrm{Gate}})_t\ \mathrm{CZ}_{c,t}\ (I \otimes H_{\mathrm{Gate}})_t, \qquad
\mathrm{CZ} = \mathrm{diag}(1, 1, 1, -1)
$$

<!-- に分解されます（標的ビットの前後に $H_{\mathrm{Gate}}$ をかけると、CZ が CX になります）。さらに C1 と同じく $H_{\mathrm{Gate}} = e^{i\pi/2}\, R_X(\pi)\, R_Y(\pi/2)$ と分解されるので、transpile 後のゲートの数は次のようになります。 -->

(applying $H_{\mathrm{Gate}}$ before and after on the target turns CZ into CX). Furthermore, as in C1, $H_{\mathrm{Gate}} = e^{i\pi/2}\, R_X(\pi)\, R_Y(\pi/2)$, so the numbers of gates after transpilation are as follows.

<!--
| 量子ビット 2 のゲート | 1 量子ビットゲート | CZ ゲート |
|---|---|---|
| $H_{\mathrm{Gate}}$ | 2 + 4 × 4 = 18 個 | 4 個 |
| $R_X(\pi/2)$, $R_Y(\pi/2)$ | 1 + 4 × 4 = 17 個 | 4 個 |
-->

| Gate on qubit 2 | 1-qubit gates | CZ gates |
|---|---|---|
| $H_{\mathrm{Gate}}$ | 2 + 4 × 4 = 18 | 4 |
| $R_X(\pi/2)$, $R_Y(\pi/2)$ | 1 + 4 × 4 = 17 | 4 |

<!-- 4 個の CZ ゲートはすべて量子ビット 2 と他の量子ビットの組なので、星形の接続グラフ（説明パート A）のまま実行でき、量子ビットを入れ替える SWAP ゲートは必要ありません。 -->

All 4 CZ gates act on pairs of qubit 2 and another qubit, so the circuit can be run on the star-shaped connectivity graph (Explanation Part A) as is, and no SWAP gates to exchange qubits are needed.

<!-- このノートブックでは transpile の最適化レベルを `optimization_level=0` に固定します。最適化によってゲートの個数や順序が変わると、ノイズを受けるゲートの数も変わってしまうからです（この回路では既定のレベルでも同じ結果になりますが、条件を固定しておきます）。 -->

In this notebook, the transpiler's optimization level is fixed at `optimization_level=0`. If optimization changed the number or order of gates, the number of gates receiving noise would also change (for this circuit the default level gives the same result, but the condition is fixed anyway).

---

<!-- ### E. ノイズモデル -->

### E. Noise model

<!-- C1 のノートブックの 1 量子ビットゲートのノイズに、CZ ゲートのノイズを加えます。すべての量子ビット・すべての CZ ゲートに同じ強さのノイズを加えます。 -->

We add CZ gate noise to the 1-qubit gate noise of the C1 notebook. Noise of the same strength is added on all qubits and all CZ gates.

<!--
| パラメータ | 論文の値 | このノートブックでの設定 |
|---|---|---|
| 1 量子ビットゲートの忠実度 | $\ge 99.9\%$ | 脱分極パラメータ $\lambda = 0.00198$（C1 と同じ） |
| CZ ゲートの忠実度 | $\ge 99.0\%$ | 2 量子ビットの脱分極パラメータ $\lambda_{\mathrm{CZ}} = 0.013245$（下で説明） |
| 読み出しの忠実度 | $\ge 97\%$ | 読み出し誤差 $\varepsilon = 0.03$（対称） |
| 1 量子ビットゲートの時間 | $\le 20$ ns | $t_g = 20$ ns |
| CZ ゲートの時間 | $\le 60$ ns | $t_{\mathrm{CZ}} = 60$ ns |
| 緩和時間 $T_1$ | 0.964 ms | $T_1 = 0.964 \times 10^6$ ns |
| 位相緩和時間 $T_2$ | 1.155 ms | $T_2 = 1.155 \times 10^6$ ns |
-->

| Parameter | Value in the paper | Setting in this notebook |
|---|---|---|
| 1-qubit gate fidelity | $\ge 99.9\%$ | Depolarizing parameter $\lambda = 0.00198$ (same as C1) |
| CZ gate fidelity | $\ge 99.0\%$ | 2-qubit depolarizing parameter $\lambda_{\mathrm{CZ}} = 0.013245$ (explained below) |
| Readout fidelity | $\ge 97\%$ | Readout error $\varepsilon = 0.03$ (symmetric) |
| 1-qubit gate time | $\le 20$ ns | $t_g = 20$ ns |
| CZ gate time | $\le 60$ ns | $t_{\mathrm{CZ}} = 60$ ns |
| Relaxation time $T_1$ | 0.964 ms | $T_1 = 0.964 \times 10^6$ ns |
| Dephasing time $T_2$ | 1.155 ms | $T_2 = 1.155 \times 10^6$ ns |

<!-- **CZ ゲートのノイズ**：CZ ゲートの後に、(a') 2 量子ビットの脱分極誤差と、(b') 2 個の量子ビットそれぞれに $t_{\mathrm{CZ}} = 60$ ns の熱緩和誤差を加えます。2 量子ビット（次元 $d = 4$）の脱分極誤差 -->

**CZ gate noise**: After each CZ gate, we add (a') a 2-qubit depolarizing error and (b') a thermal relaxation error of $t_{\mathrm{CZ}} = 60$ ns on each of the two qubits. The 2-qubit (dimension $d = 4$) depolarizing error

$$
\mathcal{E}_{\mathrm{dep}}^{(2)}(\rho) = (1 - \lambda_{\mathrm{CZ}})\,\rho + \lambda_{\mathrm{CZ}}\,\frac{I_4}{4}
$$

<!-- の平均ゲート忠実度は $1 - 3\lambda_{\mathrm{CZ}}/4$ です（1 量子ビットでは $1 - \lambda/2$ でした）。C1 と同じ考え方で、熱緩和と合わせた平均ゲート忠実度が 99.0% になるように決めます。60 ns の熱緩和誤差 2 個分の平均ゲート不忠実度は約 $6.645 \times 10^{-5}$ なので -->

has an average gate fidelity of $1 - 3\lambda_{\mathrm{CZ}}/4$ (for 1 qubit it was $1 - \lambda/2$). Following the same approach as C1, we choose it so that the average gate fidelity including thermal relaxation is 99.0%. The average gate infidelity of the two 60 ns thermal relaxation errors is about $6.645 \times 10^{-5}$, so

$$
\lambda_{\mathrm{CZ}} = \frac{4}{3}\left(0.01 - 6.645 \times 10^{-5}\right) \approx 0.013245
$$

<!-- です。5.1 で `average_gate_fidelity` を使って確かめます。 -->

This is checked using `average_gate_fidelity` in 5.1.

<!-- 加えるノイズは (a) 1 量子ビットゲートの脱分極誤差、(b) 1 量子ビットゲートの熱緩和誤差、(a') CZ ゲートの脱分極誤差、(b') CZ ゲートの熱緩和誤差、(c) 読み出し誤差です。 -->

The noise added is (a) 1-qubit gate depolarizing error, (b) 1-qubit gate thermal relaxation error, (a') CZ gate depolarizing error, (b') CZ gate thermal relaxation error, and (c) readout error.

<!--
**ノイズが結果に与える影響の予想**：
- ゲートノイズで一部の量子ビットの値がそろわなくなり、non-GHZ の結果が現れます。5.9 で使う密度行列の計算では、ゲートノイズだけで $f_{\text{non-GHZ}} \approx 0.051$（このうち CZ ゲートの寄与が大きい）、読み出し誤差だけで約 0.141、両方で約 0.184 です。
- それでも多数決で得たビットは、誤りの大部分が訂正され、偏りも統計的な揺らぎ（$N = 10^6$ で約 $4 \times 10^{-4}$）の範囲に収まると予想されます。対称な誤りは公平なビットを偏らせないからです（説明パート C）。熱緩和はわずかに 0 側へ偏らせますが、その大きさは $10^{-4}$ より小さく、この shot 数では検出できません。
-->

**Expected effects of the noise on the results**:
- Gate noise makes the values of some qubits disagree, so non-GHZ outcomes appear. In the density-matrix calculation used in 5.9, $f_{\text{non-GHZ}} \approx 0.051$ with gate noise only (of which the CZ gates are a large contribution), about 0.141 with readout error only, and about 0.184 with both.
- Even so, most of the errors in the bit from the majority vote are corrected, and the bias is expected to stay within the statistical fluctuation (about $4 \times 10^{-4}$ for $N = 10^6$). This is because symmetric errors do not bias a fair bit (Explanation Part C). Thermal relaxation biases it slightly toward 0, but the size is less than $10^{-4}$ and cannot be detected at this number of shots.

<!-- **このモデルでは再現できない実機の性質**：論文の Odra 5 の 5 量子ビット C3 回路では、1 の割合が 50.32%〜50.42% と 50% を上回りました。非対称な読み出し誤差 $P(0|1) > P(1|0)$ は 1 の割合を**下げる**方向に働くので、このノートブックのモデル（量子ビット間の干渉なし）では、50% を上回る偏りは再現できません。CZ ゲートの実行中に他の量子ビットが待っている間のノイズ（アイドル時のノイズ）も入れていません。 -->

**Properties of real hardware that this model cannot reproduce**: In the 5-qubit C3 circuit on Odra 5 in the paper, the fraction of 1s was 50.32%–50.42%, above 50%. Asymmetric readout error $P(0|1) > P(1|0)$ acts to **lower** the fraction of 1s, so the model in this notebook (which has no interference between qubits) cannot reproduce a bias above 50%. Noise on other qubits while they wait during the execution of CZ gates (idle noise) is not included either.

<!-- このノートブックでは、ノイズの種類を次の 3 つから選べます。 -->

In this notebook, the type of noise can be chosen from the following three options.

<!--
| `NOISE_KIND` | 加えるノイズ |
|---|---|
| `"gate"` | (a) + (b) と (a') + (b') のゲートノイズだけ |
| `"readout"` | (c) の読み出し誤差だけ |
| `"both"` | ゲートノイズと読み出し誤差の両方 |
-->

| `NOISE_KIND` | Noise added |
|---|---|
| `"gate"` | Only the gate noise (a) + (b) and (a') + (b') |
| `"readout"` | Only the readout error (c) |
| `"both"` | Both the gate noise and the readout error |

---

<!-- ### F. 使用する Qiskit の関数・クラス -->

### F. Qiskit functions and classes used

<!-- C1・C2 のノートブックで使ったものに加えて、次のものを使います。 -->

In addition to those used in the C1 and C2 notebooks, we use the following.

<!--
| 関数・クラス | パッケージ | 役割 |
|---|---|---|
| `qc.cx(c, t)` | `qiskit` | 制御ビット `c`、標的ビット `t` の CX（CNOT）ゲートを作用させる |
| `transpile(qc, basis_gates=..., optimization_level=0)` | `qiskit` | 最適化をせずに、指定したゲートだけを使う回路に変換する |
| `qc_t.data` | `qiskit` | 回路のゲートを順に並べたリスト。CZ ゲートがどの量子ビットに作用しているかを調べるのに使う |
| `Statevector(qc)` | `qiskit.quantum_info` | 測定前の状態ベクトルを計算する。GHZ 状態になっているか確かめる |
| `sv.probabilities_dict()` | `qiskit.quantum_info` | 状態ベクトルから各結果の確率の辞書を作る |
| `depolarizing_error(λ, 2)` | `qiskit_aer.noise` | 2 量子ビットの脱分極誤差を作る |
| `err1.tensor(err2)` | `qiskit_aer.noise` | 2 つの 1 量子ビット誤差を並べて 2 量子ビットの誤差にする（CZ の熱緩和に使う） |
| `err_a.compose(err_b)` | `qiskit_aer.noise` | 誤差 a の後に誤差 b が起こる誤差を作る |
| `noise_model.add_all_qubit_quantum_error(err, ["cz"])` | `qiskit_aer.noise` | すべての CZ ゲートの後に 2 量子ビットの誤差を加える |
| `average_gate_fidelity(err)` | `qiskit.quantum_info` | 誤差の平均ゲート忠実度を計算する（5.1 で $\lambda_{\mathrm{CZ}}$ を確かめる） |
| `ReadoutError([[1-a, a], [b, 1-b]])` | `qiskit_aer.noise` | 読み出し誤差を作る。5.9 では非対称な誤差（$a \ne b$）も使う |
-->

| Function / class | Package | Role |
|---|---|---|
| `qc.cx(c, t)` | `qiskit` | Applies a CX (CNOT) gate with control `c` and target `t` |
| `transpile(qc, basis_gates=..., optimization_level=0)` | `qiskit` | Converts a circuit into one that uses only the specified gates, without optimization |
| `qc_t.data` | `qiskit` | List of the circuit's gates in order. Used to find which qubits the CZ gates act on |
| `Statevector(qc)` | `qiskit.quantum_info` | Computes the statevector before measurement. Used to check that it is the GHZ state |
| `sv.probabilities_dict()` | `qiskit.quantum_info` | Creates a dictionary of outcome probabilities from the statevector |
| `depolarizing_error(λ, 2)` | `qiskit_aer.noise` | Creates a 2-qubit depolarizing error |
| `err1.tensor(err2)` | `qiskit_aer.noise` | Places two 1-qubit errors side by side to form a 2-qubit error (used for CZ thermal relaxation) |
| `err_a.compose(err_b)` | `qiskit_aer.noise` | Creates an error in which error b occurs after error a |
| `noise_model.add_all_qubit_quantum_error(err, ["cz"])` | `qiskit_aer.noise` | Adds a 2-qubit error after every CZ gate |
| `average_gate_fidelity(err)` | `qiskit.quantum_info` | Computes the average gate fidelity of an error (used in 5.1 to check $\lambda_{\mathrm{CZ}}$) |
| `ReadoutError([[1-a, a], [b, 1-b]])` | `qiskit_aer.noise` | Creates a readout error. Asymmetric errors ($a \ne b$) are also used in 5.9 |

<!-- 数値計算には `numpy`、グラフには `matplotlib` を使います。 -->

`numpy` is used for numerical computation and `matplotlib` for plots.

---

<!-- ### G. このノートブックの使い方 -->

### G. How to use this notebook

<!-- 上のセルから順に実行してください。シミュレーションの条件は、次の変数を書き換えて選びます。 -->

Run the cells in order from the top. The simulation conditions are selected by editing the following variables.

<!--
| 変数 | 意味 | 選択肢・既定値 |
|---|---|---|
| `SAVE_RESULTS` | 4.3〜5.8 の raw data と結果を保存するか（1. のセル） | `True`（`results/c3/<日時>/` に保存）または `False`（保存しない。既定） |
| `SAVE_SWEEP` | 5.9 のノイズの強さを変える実験の raw data と結果を保存するか（1. のセル） | `True`（同じ日時のフォルダに保存）または `False`（保存しない。既定） |
| `GATE_IDEAL` | ノイズなしで量子ビット 2 に使うゲート | `"H"`, `"Rx"`, `"Ry"`（既定 `"H"`） |
| `SHOTS_IDEAL` | ノイズなしの測定回数（= 乱数のビット数） | 既定 `1_000_000` |
| `GATE_NOISY` | ノイズありで量子ビット 2 に使うゲート | `"H"`, `"Rx"`, `"Ry"`（既定 `"H"`） |
| `NOISE_KIND` | 加えるノイズの種類 | `"gate"`, `"readout"`, `"both"`（既定 `"both"`） |
| `SHOTS_NOISY` | ノイズありの測定回数 | 既定 `1_000_000` |
| `N_TRIALS` | 繰り返しの回数 | 既定 `10` |
| `SEED` | シミュレータの乱数シード | `None`（毎回違う結果）または整数（結果を再現できる） |
-->

| Variable | Meaning | Options / default |
|---|---|---|
| `SAVE_RESULTS` | Whether to save the raw data and results of 4.3–5.8 (cell in 1.) | `True` (save in `results/c3/<date_time>/`) or `False` (do not save; default) |
| `SAVE_SWEEP` | Whether to save the raw data and results of the noise-strength sweep in 5.9 (cell in 1.) | `True` (save in the same date-and-time folder) or `False` (do not save; default) |
| `GATE_IDEAL` | Gate applied to qubit 2 in the noiseless case | `"H"`, `"Rx"`, `"Ry"` (default `"H"`) |
| `SHOTS_IDEAL` | Number of noiseless measurements (= number of random bits) | Default `1_000_000` |
| `GATE_NOISY` | Gate applied to qubit 2 in the noisy case | `"H"`, `"Rx"`, `"Ry"` (default `"H"`) |
| `NOISE_KIND` | Type of noise added | `"gate"`, `"readout"`, `"both"` (default `"both"`) |
| `SHOTS_NOISY` | Number of noisy measurements | Default `1_000_000` |
| `N_TRIALS` | Number of repetitions | Default `10` |
| `SEED` | Random seed of the simulator | `None` (different results each time) or an integer (results are reproducible) |

<!-- 測定回数の既定値は、乱数のビット数が論文と同じ $10^6$ ビットになるようにしています。 -->

The default number of measurements is set so that the number of random bits is $10^6$, the same as in the paper.

<!-- #### 保存されるファイル -->

#### Saved files

<!-- 1. のセルで `SAVE_RESULTS = True` または `SAVE_SWEEP = True` にすると、そのセルを実行した日時の名前のフォルダ `results/c3/<年-月-日_時分秒>/`（変数 `RUN_DIR`）を作ります。`SAVE_RESULTS = True` なら 4.3〜5.8 の raw data と結果を、`SAVE_SWEEP = True` なら 5.9 の raw data と結果を、その中に保存します。5.9 は保存量が多いので、別のスイッチにしています。どちらも `False`（既定）なら何も保存しません。1. のセルを実行し直すたびに新しいフォルダになるので、前の結果を上書きすることはありません。 -->

If you set `SAVE_RESULTS = True` or `SAVE_SWEEP = True` in the cell in 1., a folder named after the date and time that cell was run, `results/c3/<year-month-day_hourminutesecond>/` (variable `RUN_DIR`), is created. `SAVE_RESULTS = True` saves the raw data and results of 4.3–5.8 in it, and `SAVE_SWEEP = True` saves those of 5.9. Because 5.9 produces a lot of data, it has its own switch. If both are `False` (the default), nothing is saved. Each time you rerun the cell in 1., a new folder is used, so earlier results are never overwritten.

<!-- `RUN_DIR` の中は、実験（節）ごとのフォルダに分かれています。フォルダ名の先頭の数字は、ノートブックの節の番号です。 -->

Inside `RUN_DIR`, there is one folder per experiment (section). The number at the start of each folder name is the section number in this notebook.

<!--
| フォルダ | 節 | 中身 |
|---|---|---|
| `4.3_ideal_single/` | 4.3 | ノイズなしの 1 回の測定の raw data |
| `4.5_ideal_trials/` | 4.5 | ノイズなしの繰り返し測定の raw data（試行ごとに 1 ファイル）と、全試行の一覧 `trials.csv` |
| `4.6_ideal_gates/` | 4.6 | 3 つのゲートの raw data と、その一覧 `summary.csv` |
| `5.5_noisy_single/` | 5.5 | ノイズありの 1 回の測定の raw data |
| `5.7_noisy_trials/` | 5.7 | ノイズありの繰り返し測定の raw data（試行ごとに 1 ファイル）と `trials.csv` |
| `5.8_noisy_comparison/` | 5.8 | ゲート × ノイズの種類の 9 通りの raw data と `summary.csv` |
| `5.9_noise_sweep/` | 5.9 | ノイズの強さを変えた各条件・各試行の raw data（ノイズの種類ごとのフォルダ）と、`trials.csv`・`summary.csv`（`SAVE_SWEEP = True` のときだけ） |
-->

| Folder | Section | Contents |
|---|---|---|
| `4.3_ideal_single/` | 4.3 | Raw data of the single noiseless measurement |
| `4.5_ideal_trials/` | 4.5 | Raw data of the repeated noiseless measurements (one file per trial) and the list of all trials, `trials.csv` |
| `4.6_ideal_gates/` | 4.6 | Raw data for the three gates and their list, `summary.csv` |
| `5.5_noisy_single/` | 5.5 | Raw data of the single noisy measurement |
| `5.7_noisy_trials/` | 5.7 | Raw data of the repeated noisy measurements (one file per trial) and `trials.csv` |
| `5.8_noisy_comparison/` | 5.8 | Raw data for the 9 combinations of gate × noise type and `summary.csv` |
| `5.9_noise_sweep/` | 5.9 | Raw data for each condition and trial with varying noise strength (one folder per noise type), `trials.csv`, and `summary.csv` (only when `SAVE_SWEEP = True`) |

<!-- 既定の設定で両方を保存すると、次のようになります。 -->

With the default settings and both switches on, the saved files look like this.

<!--
```
results/c3/
└── 2026-11-10_153012/                 RUN_DIR（1. のセルを実行した日時）
    ├── 4.3_ideal_single/
    │   ├── c3_H_ideal.txt
    │   ├── c3_H_ideal.shots.txt
    │   └── c3_H_ideal.txt.json
    ├── 4.5_ideal_trials/
    │   ├── c3_H_ideal_trial001.txt
    │   ├── c3_H_ideal_trial001.shots.txt
    │   ├── c3_H_ideal_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 4.6_ideal_gates/
    │   ├── c3_H_ideal.txt
    │   ├── c3_H_ideal.shots.txt
    │   ├── c3_H_ideal.txt.json
    │   ├── c3_Rx_ideal.txt ...
    │   └── summary.csv
    ├── 5.5_noisy_single/
    │   ├── c3_H_noise-both.txt
    │   ├── c3_H_noise-both.shots.txt
    │   └── c3_H_noise-both.txt.json
    ├── 5.7_noisy_trials/
    │   ├── c3_H_noise-both_trial001.txt
    │   ├── c3_H_noise-both_trial001.shots.txt
    │   ├── c3_H_noise-both_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 5.8_noisy_comparison/
    │   ├── c3_H_noise-gate.txt
    │   ├── c3_H_noise-gate.shots.txt
    │   ├── c3_H_noise-gate.txt.json
    │   ├── c3_H_noise-readout.txt ...
    │   └── summary.csv
    └── 5.9_noise_sweep/
        ├── trials.csv
        ├── summary.csv
        ├── depolarizing/
        │   ├── c3_H_depolarizing-0_trial001.txt
        │   ├── c3_H_depolarizing-0_trial001.shots.txt
        │   ├── c3_H_depolarizing-0_trial001.txt.json
        │   └── c3_H_depolarizing-0.01_trial001.txt ...
        └── ...
```
-->

```
results/c3/
└── 2026-11-10_153012/                 RUN_DIR (date and time the cell in 1. was run)
    ├── 4.3_ideal_single/
    │   ├── c3_H_ideal.txt
    │   ├── c3_H_ideal.shots.txt
    │   └── c3_H_ideal.txt.json
    ├── 4.5_ideal_trials/
    │   ├── c3_H_ideal_trial001.txt
    │   ├── c3_H_ideal_trial001.shots.txt
    │   ├── c3_H_ideal_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 4.6_ideal_gates/
    │   ├── c3_H_ideal.txt
    │   ├── c3_H_ideal.shots.txt
    │   ├── c3_H_ideal.txt.json
    │   ├── c3_Rx_ideal.txt ...
    │   └── summary.csv
    ├── 5.5_noisy_single/
    │   ├── c3_H_noise-both.txt
    │   ├── c3_H_noise-both.shots.txt
    │   └── c3_H_noise-both.txt.json
    ├── 5.7_noisy_trials/
    │   ├── c3_H_noise-both_trial001.txt
    │   ├── c3_H_noise-both_trial001.shots.txt
    │   ├── c3_H_noise-both_trial001.txt.json
    │   ├── ...
    │   └── trials.csv
    ├── 5.8_noisy_comparison/
    │   ├── c3_H_noise-gate.txt
    │   ├── c3_H_noise-gate.shots.txt
    │   ├── c3_H_noise-gate.txt.json
    │   ├── c3_H_noise-readout.txt ...
    │   └── summary.csv
    └── 5.9_noise_sweep/
        ├── trials.csv
        ├── summary.csv
        ├── depolarizing/
        │   ├── c3_H_depolarizing-0_trial001.txt
        │   ├── c3_H_depolarizing-0_trial001.shots.txt
        │   ├── c3_H_depolarizing-0_trial001.txt.json
        │   └── c3_H_depolarizing-0.01_trial001.txt ...
        └── ...
```

<!--
raw data のファイル名は `<回路>_<ゲート>_<条件>[_trial<試行番号>].txt` です。

- 回路：`c3`
- ゲート：`H`, `Rx`, `Ry`
- 条件：`ideal`（ノイズなし）、`noise-gate`・`noise-readout`・`noise-both`（5.3 の `NOISE_KIND`）、5.9 では `<ノイズの種類>-<強さ>`（例：`readout-0.04`、`thermal-2500`）。5.9 のノイズの種類は `depolarizing`, `cz_depolarizing`, `thermal`, `readout`, `readout_asym` です
- 試行番号：繰り返し測定（4.5・5.7・5.9）だけにつく、1 から数えた 3 桁の番号
-->

Raw-data files are named `<circuit>_<gate>_<condition>[_trial<trial number>].txt`.

- Circuit: `c3`
- Gate: `H`, `Rx`, `Ry`
- Condition: `ideal` (no noise); `noise-gate`, `noise-readout`, `noise-both` (`NOISE_KIND` in 5.3); in 5.9, `<noise type>-<strength>` (e.g., `readout-0.04`, `thermal-2500`). The noise types in 5.9 are `depolarizing`, `cz_depolarizing`, `thermal`, `readout`, `readout_asym`
- Trial number: only for repeated measurements (4.5, 5.7, 5.9); a 3-digit number counted from 1

<!--
| ファイル | 中身 |
|---|---|
| `<名前>.txt` | `'0'` と `'1'` の文字を改行なしで並べたテキスト（長さ $N$）。各 shot の多数決のビットを測定順に並べたもの。SP 800-90B や抽出器に渡すのはこのファイル。SP800_90B・Toeplitz・Trevisan・SP800_22 のノートブックの `load_bits` で読める |
| `<名前>.shots.txt` | 各 shot の 5 ビットの文字列（Qiskit の順：量子ビット 4, 3, 2, 1, 0）を 1 行に 1 つずつ、測定順に並べたテキスト |
| `<名前>.txt.json` | 測定の条件と結果。キーは `circuit`, `gate`, `shots`, `seed`, `metadata`（条件・試行番号など）, `decoding`, `shots_bit_order`, `basis_gates`, `optimization_level`, `method`, `qiskit_version`, `qiskit_aer_version`, `noise_model`（`NoiseModel.to_dict` の内容。ノイズなしなら `null`）, `counts`, `evaluation`（`evaluate_c3` の戻り値） |
| `trials.csv`（4.5・5.7） | 1 試行 1 行。列は `file`（raw data のファイル名）, `gate`, `condition`, `trial`, `seed` と、`evaluate_c3` の戻り値のうち数値の項目 |
| `summary.csv`（4.6・5.8） | 1 ファイル 1 行。列は `file`, `gate`, `condition`, `seed` と、`evaluate_c3` の戻り値のうち数値の項目 |
| `5.9_noise_sweep/trials.csv` | 1 条件・1 試行 1 行。列は `component`, `strength`, `gate`, `trial`, `seed`, `shots`, `P1`, `bias`, `shannon`, `min_entropy`, `non_ghz`, `raw_min_entropy` |
| `5.9_noise_sweep/summary.csv` | 1 条件 1 行。列は `component`, `strength`, `gate`、密度行列から求めた `exact_P1`, `exact_bias`, `exact_min_entropy`, `exact_non_ghz`、`SWEEP_METRICS` の各指標の試行間の平均 `<指標>_mean` と標準偏差 `<指標>_std` |
-->

| File | Contents |
|---|---|
| `<name>.txt` | Text of the characters `'0'` and `'1'` with no line breaks (length $N$): the majority-vote bit of each shot in measurement order. This is the file to pass to SP 800-90B and the extractors. Can be read with `load_bits` in the SP800_90B, Toeplitz, Trevisan, and SP800_22 notebooks |
| `<name>.shots.txt` | Text with the 5-bit string of each shot (Qiskit order: qubits 4, 3, 2, 1, 0), one per line, in measurement order |
| `<name>.txt.json` | Measurement conditions and results. Keys: `circuit`, `gate`, `shots`, `seed`, `metadata` (condition, trial number, and so on), `decoding`, `shots_bit_order`, `basis_gates`, `optimization_level`, `method`, `qiskit_version`, `qiskit_aer_version`, `noise_model` (the contents of `NoiseModel.to_dict`; `null` without noise), `counts`, `evaluation` (the return value of `evaluate_c3`) |
| `trials.csv` (4.5, 5.7) | One row per trial. Columns: `file` (the raw-data file name), `gate`, `condition`, `trial`, `seed`, and the numeric items of the return value of `evaluate_c3` |
| `summary.csv` (4.6, 5.8) | One row per file. Columns: `file`, `gate`, `condition`, `seed`, and the numeric items of the return value of `evaluate_c3` |
| `5.9_noise_sweep/trials.csv` | One row per condition and trial. Columns: `component`, `strength`, `gate`, `trial`, `seed`, `shots`, `P1`, `bias`, `shannon`, `min_entropy`, `non_ghz`, `raw_min_entropy` |
| `5.9_noise_sweep/summary.csv` | One row per condition. Columns: `component`, `strength`, `gate`; `exact_P1`, `exact_bias`, `exact_min_entropy`, `exact_non_ghz` computed from the density matrix; and the mean `<metric>_mean` and standard deviation `<metric>_std` over trials of each of the `SWEEP_METRICS` |

<!--
- ファイルは上書きしません。同じ `RUN_DIR` のまま同じセルをもう一度実行すると、測定を始める前にエラー（`FileExistsError`）になります。もう一度保存するときは、1. のセルを実行し直して新しいフォルダを作ってください。
- CSV は、その実験のすべての試行・条件が終わったあとに、raw data と同じフォルダに書きます。CSV がないフォルダは、途中で止まった実験の raw data です。
- `run_c3(..., raw_path=...)` や `run_trials(..., raw_dir=...)` に自分で保存先を指定した場合は、`SAVE_RESULTS` に関係なくその場所に保存します（4.5 を参照）。
- 保存量の目安：1 組（`.txt` 約 1 MB と `.shots.txt` 約 6 MB）が約 7 MB です。既定の設定では、4.3〜5.8 の合計が約 240 MB、5.9（`SAVE_SWEEP = True` のとき）が約 3.3 GB です。5.9 を保存するときは `SWEEP_SHOTS` を小さくすることも検討してください。
-->

- Files are never overwritten. If you run the same cell again with the same `RUN_DIR`, an error (`FileExistsError`) is raised before any measurement. To save again, rerun the cell in 1. to create a new folder.
- The CSV files are written in the same folder as the raw data, after all trials and conditions of that experiment have finished. A folder without CSV files holds the raw data of an experiment that stopped partway.
- If you give your own destination to `run_c3(..., raw_path=...)` or `run_trials(..., raw_dir=...)`, the files are saved there regardless of `SAVE_RESULTS` (see 4.5).
- Approximate size: Each set (`.txt` of about 1 MB and `.shots.txt` of about 6 MB) is about 7 MB. With the default settings, 4.3–5.8 save about 240 MB in total, and 5.9 (when `SAVE_SWEEP = True`) saves about 3.3 GB. When saving 5.9, consider reducing `SWEEP_SHOTS`.

<!-- ## 1. 準備 -->

## 1. Setup

<!-- 使うライブラリを読み込み、ネイティブゲート（説明パート D）を `BASIS_GATES` として定義します。 -->

We load the libraries used and define the native gates (Explanation Part D) as `BASIS_GATES`.

<!--
- `qiskit`：回路を作る・transpile する・状態ベクトルや平均ゲート忠実度を計算する
- `qiskit_aer`：シミュレータとノイズモデル
- `numpy`：5.9 で確率の計算に使う
- `IPython.display`：結果の表を Markdown で表示する
- `matplotlib`：5.9 のグラフを描く
-->

- `qiskit`: building and transpiling circuits, and computing statevectors and average gate fidelities
- `qiskit_aer`: simulator and noise models
- `numpy`: used for probability calculations in 5.9
- `IPython.display`: displaying result tables as Markdown
- `matplotlib`: drawing the plots in 5.9

<!-- `CONTROL` は重ね合わせを作る中心の量子ビット（2）、`TARGETS` は CX ゲートをかける順に並べた標的ビット（論文の Fig. 4 と同じ 1, 3, 0, 4）です。 -->

`CONTROL` is the central qubit (2) on which the superposition is created, and `TARGETS` lists the target qubits in the order the CX gates are applied (1, 3, 0, 4, the same as Fig. 4 of the paper).

<!-- **結果の保存**：このセルの 2 つのスイッチで、raw data と結果を保存するかどうかを選びます。`SAVE_RESULTS = True` で 4.3〜5.8 を、`SAVE_SWEEP = True` で 5.9（ノイズの強さを段階的に変える実験。保存量が多い）を保存します。どちらかを `True` にすると、このセルを実行した日時の名前のフォルダ（例：`results/c3/2026-11-10_153012/`）を作り、その中の実験ごとのフォルダに保存します。どちらも `False`（既定）なら何も保存しません。保存先と中身は「G. このノートブックの使い方」の「保存されるファイル」を参照してください。 -->

**Saving the results**: the two switches in this cell select whether the raw data and results are saved. `SAVE_RESULTS = True` saves 4.3–5.8, and `SAVE_SWEEP = True` saves 5.9 (the experiment that changes the noise strength step by step, which produces a lot of data). If either is `True`, a folder named after the date and time this cell was run (e.g., `results/c3/2026-11-10_153012/`) is created, and the files are saved in per-experiment folders inside it. If both are `False` (the default), nothing is saved. For the destinations and contents, see "Saved files" in "G. How to use this notebook".

<!-- セルは「回路の定数」「シミュレーションのシード」「保存のスイッチ」の 3 つのまとまりに分けています。保存に使う関数は、次の「保存に使う関数」のセルにまとめています。 -->

The cell is divided into three groups: circuit constants, the simulation seed, and the saving switches. The functions used for saving are collected in the following "Functions used for saving" cell.

In [ ]:
import csv
import json
import math
import statistics
from collections import Counter
from datetime import datetime
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import qiskit
import qiskit_aer
from IPython.display import Markdown, display
from qiskit import QuantumCircuit, transpile
from qiskit.quantum_info import Operator, Statevector, average_gate_fidelity
from qiskit_aer import AerSimulator
from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error, thermal_relaxation_error

print("qiskit     :", qiskit.__version__)
print("qiskit-aer :", qiskit_aer.__version__)

# ---- Circuit ----
BASIS_GATES = ["rx", "ry", "cz"]                # Native gates of IQM Spark (Odra 5) (arXiv:2512.09862, Table II)
GATES = ["H", "Rx", "Ry"]                       # Gates that create the superposition
N_QUBITS = 5
CONTROL = 2                                     # Central qubit
TARGETS = [1, 3, 0, 4]                          # Target qubits in the order the CX gates are applied (Fig. 4 of the paper)
MAJORITY = N_QUBITS // 2 + 1                    # Minimum number of 1s for the majority vote to give 1 (3 of 5)
GHZ_STATES = ("0" * N_QUBITS, "1" * N_QUBITS)   # The two outcomes of the GHZ state

# ---- Simulation ----
SEED = None    # Random seed of the simulator (None: different results each time; integer: reproducible)

# ---- Saving (see "Saved files" in G) ----
SAVE_RESULTS = False    # Save the raw data and results of 4.3-5.8
SAVE_SWEEP = False      # Save the raw data and results of the noise-strength sweep in 5.9 (large)
RESULTS_ROOT = Path("results/c3")
RUN_DIR = RESULTS_ROOT / datetime.now().strftime("%Y-%m-%d_%H%M%S") if (SAVE_RESULTS or SAVE_SWEEP) else None

if RUN_DIR is None:
    print("save to    : nothing is saved")
else:
    print(f"save to    : {RUN_DIR} (4.3-5.8: {SAVE_RESULTS}, 5.9: {SAVE_SWEEP})")

<!-- **保存に使う関数** -->

**Functions used for saving**

<!-- raw data と CSV の保存は、次のセルの関数にまとめています。`run_c3`（4.5）や 4.3・5.5 のセルはこれらを呼び出すだけです。どのファイルも上書きしません。 -->

Saving the raw data and the CSV files is collected in the functions of the next cell. `run_c3` (4.5) and the cells in 4.3 and 5.5 only call them. No file is ever overwritten.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `output_dir` | 4.3〜5.8 の実験のフォルダ `RUN_DIR/<name>`（`SAVE_RESULTS = False` なら `None`） | `output_dir("4.5_ideal_trials")` |
| `raw_files` | 1 組の raw data の 3 つのファイルのパス（`.txt`, `.shots.txt`, `.txt.json`） | `raw_files("results/c3/manual/c3_H_ideal.txt")` |
| `ensure_new` | 3 つのファイルのどれかがすでにあればエラー（測定の前に確かめるため） | `ensure_new("results/c3/manual/c3_H_ideal.txt")` |
| `save_raw` | 各 shot の 5 ビットの文字列から、多数決のビット列・5 ビットの文字列・条件の JSON を保存し、`.txt` のパスを返す | `save_raw(path, result.get_memory(), "H", 1000, 2026)` |
| `write_csv` | 1 レコード 1 行の CSV を書き、パスを返す（数値と文字列の項目だけを列にする） | `write_csv(path, records)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `output_dir` | Folder of an experiment in 4.3–5.8, `RUN_DIR/<name>` (`None` when `SAVE_RESULTS = False`) | `output_dir("4.5_ideal_trials")` |
| `raw_files` | Paths of the three files of one raw-data set (`.txt`, `.shots.txt`, `.txt.json`) | `raw_files("results/c3/manual/c3_H_ideal.txt")` |
| `ensure_new` | Raises an error if any of the three files already exists (checked before measuring) | `ensure_new("results/c3/manual/c3_H_ideal.txt")` |
| `save_raw` | From the 5-bit string of each shot, saves the majority-vote bits, the 5-bit strings, and the JSON of conditions, and returns the path of the `.txt` | `save_raw(path, result.get_memory(), "H", 1000, 2026)` |
| `write_csv` | Writes a CSV with one row per record and returns its path (only numeric and string items become columns) | `write_csv(path, records)` |

<!-- `save_raw` の引数 `gate`, `shots`, `seed`, `noise_model`, `method`, `optimization_level`, `metadata` は、JSON に記録する測定の条件です。 -->

The arguments `gate`, `shots`, `seed`, `noise_model`, `method`, `optimization_level`, and `metadata` of `save_raw` are the measurement conditions recorded in the JSON.

In [ ]:
def output_dir(name):
    # Folder of one experiment in 4.3-5.8 (None when SAVE_RESULTS = False)
    return RUN_DIR / name if SAVE_RESULTS else None


def raw_files(path):
    # The three files of one raw-data set: bits from the majority vote, 5-bit strings, conditions
    path = Path(path)
    return path, path.with_suffix(".shots.txt"), path.with_name(path.name + ".json")


def ensure_new(path):
    # Files are never overwritten: stop if any of the three files already exists
    for p in raw_files(path):
        if p.exists():
            raise FileExistsError(f"Destination already exists: {p}. Run the Setup cell (1.) again to save in a new folder.")


def save_raw(path, memory, gate, shots, seed, noise_model=None, *,
             method="automatic", optimization_level=0, metadata=None):
    # memory: the 5-bit string of each shot in measurement order (result.get_memory())
    ensure_new(path)
    bits_path, shots_path, info_path = raw_files(path)
    counts = dict(Counter(memory))
    info = {
        "circuit": "C3", "gate": gate, "shots": shots, "seed": seed, "metadata": metadata,
        "decoding": "majority vote (number of 1s >= 3 -> 1)",
        "shots_bit_order": "each line: q4 q3 q2 q1 q0 (Qiskit order), lines in measurement order",
        "basis_gates": BASIS_GATES, "optimization_level": optimization_level, "method": method,
        "qiskit_version": qiskit.__version__, "qiskit_aer_version": qiskit_aer.__version__,
        "noise_model": None if noise_model is None else noise_model.to_dict(serializable=True),
        "counts": counts, "evaluation": evaluate_c3(counts),
    }
    bits_path.parent.mkdir(parents=True, exist_ok=True)
    bits_path.write_text("".join(decode_c3(s) for s in memory), encoding="ascii")
    shots_path.write_text("\n".join(memory) + "\n", encoding="ascii")
    info_path.write_text(json.dumps(info, indent=2) + "\n", encoding="utf-8")
    return bits_path


def write_csv(path, records):
    # One row per record; only scalar values become columns. Existing files are never overwritten.
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    columns = [k for k, v in records[0].items() if isinstance(v, (str, int, float, bool, type(None)))]
    with path.open("x", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction="ignore")
        writer.writeheader()
        writer.writerows(records)
    return path

<!-- ## 2. 評価指標を計算する関数 -->

## 2. Functions for computing the evaluation metrics

<!-- 説明パート B・C の多数決と評価指標を関数にします。 -->

We turn the majority vote and evaluation metrics of Explanation Parts B and C into functions.

<!--
- `shannon_entropy(probs)`, `min_entropy(probs)`：確率のリストからエントロピーを計算する（C2 のノートブックと同じ）。多数決で得たビットには `[P0, P1]` を、5 ビットの結果には 32 通りの確率を渡します。
- `hamming_weight(bitstring)`：5 ビットの文字列に含まれる 1 の個数 $|x|$
- `decode_c3(bitstring)`：多数決 $d(x)$。1 が 3 個以上なら `"1"`、そうでなければ `"0"`
- `evaluate_c3(counts)`：5 ビットの測定回数の辞書から、(1) 多数決で得たビットの $\hat{P}(1)$・bias・$H_{\mathrm{Shannon}}$・$H_{\min}$、(2) non-GHZ の割合とハミング重みの分布、(3) 5 ビットの結果の min-entropy、(4) 事後選択の $\hat{P}_{\mathrm{ps}}(1)$ をまとめて計算する
- `expected_bias(n_bits)`：公平なビットを `n_bits` 個測ったときの bias の目安 $\sqrt{2/\pi}\,/\,(2\sqrt{n})$
- `majority_error_prob(e)`：各量子ビットが確率 `e` で読み間違えるときに多数決が誤る確率 $M(e)$
- `readout_non_ghz(e)`：読み出し誤差だけのときの non-GHZ の割合の理論値 $1 - (1-e)^5 - e^5$
-->

- `shannon_entropy(probs)`, `min_entropy(probs)`: compute entropies from a list of probabilities (same as in the C2 notebook). Pass `[P0, P1]` for the bit from the majority vote and the 32 probabilities for the 5-bit outcomes.
- `hamming_weight(bitstring)`: the number of 1s, $|x|$, in a 5-bit string
- `decode_c3(bitstring)`: majority vote $d(x)$. `"1"` if there are 3 or more 1s, otherwise `"0"`
- `evaluate_c3(counts)`: computes together, from the 5-bit measurement-count dictionary, (1) $\hat{P}(1)$, bias, $H_{\mathrm{Shannon}}$, and $H_{\min}$ of the bit from the majority vote, (2) the non-GHZ fraction and the Hamming-weight distribution, (3) the min-entropy of the 5-bit outcomes, and (4) the postselected $\hat{P}_{\mathrm{ps}}(1)$
- `expected_bias(n_bits)`: the reference bias $\sqrt{2/\pi}\,/\,(2\sqrt{n})$ when `n_bits` fair bits are measured
- `majority_error_prob(e)`: the probability $M(e)$ that the majority vote fails when each qubit is misread with probability `e`
- `readout_non_ghz(e)`: the theoretical non-GHZ fraction $1 - (1-e)^5 - e^5$ with readout error only

<!-- 多数決の結果は 5 ビットの文字列だけで決まるので、評価には `get_counts()` の集計結果があれば十分です。 -->

The result of the majority vote is determined solely by the 5-bit string, so the aggregated results from `get_counts()` are sufficient for the evaluation.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `shannon_entropy` | Shannon エントロピー（float、単位はビット） | `shannon_entropy([0.5, 0.5])` |
| `min_entropy` | min-entropy（float、単位はビット） | `min_entropy([0.5, 0.5])` |
| `hamming_weight` | 1 の個数（int） | `hamming_weight("01011")` |
| `decode_c3` | 多数決の結果（文字列 `"0"` または `"1"`） | `decode_c3("01000")` |
| `evaluate_c3` | 評価結果をまとめた辞書 | `evaluate_c3({"00000": 5, "11111": 4, "01000": 1})` |
| `expected_bias` | 公平なビットの bias の期待値の近似（float） | `expected_bias(1_000_000)` |
| `majority_error_prob` | 多数決が誤る確率 $M(e)$（float） | `majority_error_prob(0.03)` |
| `readout_non_ghz` | 読み出し誤差だけのときの non-GHZ の割合の理論値（float） | `readout_non_ghz(0.03)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `shannon_entropy` | Shannon entropy (float, in bits) | `shannon_entropy([0.5, 0.5])` |
| `min_entropy` | Min-entropy (float, in bits) | `min_entropy([0.5, 0.5])` |
| `hamming_weight` | Number of 1s (int) | `hamming_weight("01011")` |
| `decode_c3` | Result of the majority vote (the string `"0"` or `"1"`) | `decode_c3("01000")` |
| `evaluate_c3` | Dictionary collecting the evaluation results | `evaluate_c3({"00000": 5, "11111": 4, "01000": 1})` |
| `expected_bias` | Approximate expected bias of fair bits (float) | `expected_bias(1_000_000)` |
| `majority_error_prob` | Probability $M(e)$ that the majority vote fails (float) | `majority_error_prob(0.03)` |
| `readout_non_ghz` | Theoretical non-GHZ fraction with readout error only (float) | `readout_non_ghz(0.03)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>shannon_entropy</code></td><td><code>probs</code></td><td>確率のリスト。各値は 0〜1 で、合計が 1</td><td>必須</td></tr>
<tr><td><code>min_entropy</code></td><td><code>probs</code></td><td>確率のリスト。合計が 1</td><td>必須</td></tr>
<tr><td><code>hamming_weight</code></td><td><code>bitstring</code></td><td>0 と 1 からなる文字列（例：<code>"01011"</code>）</td><td>必須</td></tr>
<tr><td><code>decode_c3</code></td><td><code>bitstring</code></td><td>5 ビットの測定結果の文字列（例：<code>"01000"</code>）</td><td>必須</td></tr>
<tr><td><code>evaluate_c3</code></td><td><code>counts</code></td><td>5 ビットの測定集計の辞書。キーは <code>"00000"</code>〜<code>"11111"</code>、値は非負の数。合計は 0 より大きいこと。出なかったキーは省略可</td><td>必須</td></tr>
<tr><td><code>expected_bias</code></td><td><code>n_bits</code></td><td>ビット数。1 以上の整数</td><td>必須</td></tr>
<tr><td><code>majority_error_prob</code></td><td><code>e</code></td><td>1 個の量子ビットを読み間違える確率。0〜1</td><td>必須</td></tr>
<tr><td><code>readout_non_ghz</code></td><td><code>e</code></td><td>1 個の量子ビットを読み間違える確率。0〜1</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>shannon_entropy</code></td><td><code>probs</code></td><td>List of probabilities. Each value is from 0 to 1 and the total is 1</td><td>Required</td></tr>
<tr><td><code>min_entropy</code></td><td><code>probs</code></td><td>List of probabilities. The total is 1</td><td>Required</td></tr>
<tr><td><code>hamming_weight</code></td><td><code>bitstring</code></td><td>String of 0s and 1s (e.g., <code>"01011"</code>)</td><td>Required</td></tr>
<tr><td><code>decode_c3</code></td><td><code>bitstring</code></td><td>5-bit measurement-result string (e.g., <code>"01000"</code>)</td><td>Required</td></tr>
<tr><td><code>evaluate_c3</code></td><td><code>counts</code></td><td>Dictionary of 5-bit measurement tallies. Keys are <code>"00000"</code> to <code>"11111"</code>; values are non-negative numbers whose total is greater than 0. Keys that did not occur may be omitted</td><td>Required</td></tr>
<tr><td><code>expected_bias</code></td><td><code>n_bits</code></td><td>Number of bits. An integer of at least 1</td><td>Required</td></tr>
<tr><td><code>majority_error_prob</code></td><td><code>e</code></td><td>Probability of misreading one qubit. From 0 to 1</td><td>Required</td></tr>
<tr><td><code>readout_non_ghz</code></td><td><code>e</code></td><td>Probability of misreading one qubit. From 0 to 1</td><td>Required</td></tr>
</tbody>
</table>

<br>

<!-- `evaluate_c3` の戻り値の主なキーは次のとおりです。 -->

The main keys of the return value of `evaluate_c3` are as follows.

<!--
| キー | 意味 |
|---|---|
| `shots` | shot 数 $N$（= 多数決で得たビットの数） |
| `n0`, `n1`, `P0`, `P1`, `bias`, `shannon`, `min_entropy` | (1) 多数決で得たビットの 0/1 の個数・確率・bias・エントロピー |
| `non_ghz` | (2) $\lvert 00000\rangle$, $\lvert 11111\rangle$ 以外の結果の割合 |
| `weight_counts` | (2) 1 の個数が 0, 1, …, 5 個だった shot の数のリスト |
| `raw_min_entropy` | (3) 5 ビットの結果の分布の min-entropy（理想は 1） |
| `postselected_P1` | (4) $\lvert 00000\rangle$ と $\lvert 11111\rangle$ だけを使ったときの 1 の割合 |
-->

| Key | Meaning |
|---|---|
| `shots` | Number of shots $N$ (= number of bits from the majority vote) |
| `n0`, `n1`, `P0`, `P1`, `bias`, `shannon`, `min_entropy` | (1) Numbers of 0s and 1s, probabilities, bias, and entropies of the bits from the majority vote |
| `non_ghz` | (2) Fraction of outcomes other than $\lvert 00000\rangle$ and $\lvert 11111\rangle$ |
| `weight_counts` | (2) List of the numbers of shots with 0, 1, …, 5 ones |
| `raw_min_entropy` | (3) Min-entropy of the distribution of 5-bit outcomes (ideally 1) |
| `postselected_P1` | (4) Fraction of 1s when only $\lvert 00000\rangle$ and $\lvert 11111\rangle$ are used |


In [ ]:
def shannon_entropy(probs):
    # Treat 0 log 0 as 0
    return -sum(p * math.log2(p) for p in probs if p > 0)


def min_entropy(probs):
    return -math.log2(max(probs))


def hamming_weight(bitstring):
    return bitstring.count("1")


def decode_c3(bitstring):
    # With 5 qubits, ties do not occur
    return "1" if hamming_weight(bitstring) >= MAJORITY else "0"


def evaluate_c3(counts):
    shots = sum(counts.values())

    # (2) Number of shots for each number of 1s
    weight_counts = [0] * (N_QUBITS + 1)
    for x, n in counts.items():
        weight_counts[hamming_weight(x)] += n

    # (1) Number of shots where the majority vote gave 1 (5.9 passes probabilities as counts, so compute P(1) from the value before rounding)
    n1_value = sum(weight_counts[MAJORITY:])
    P1 = n1_value / shots
    P0 = 1 - P1

    # (2), (4) The two GHZ outcomes
    n_zero = counts.get(GHZ_STATES[0], 0)
    n_one = counts.get(GHZ_STATES[1], 0)

    return {
        "shots": shots,
        "n0": int(round(shots - n1_value)),
        "n1": int(round(n1_value)),
        "P0": P0,
        "P1": P1,
        "bias": abs(P1 - 1 / 2),
        "shannon": shannon_entropy([P0, P1]),
        "min_entropy": min_entropy([P0, P1]),
        "non_ghz": 1 - (n_zero + n_one) / shots,
        "weight_counts": weight_counts,
        "raw_min_entropy": min_entropy([n / shots for n in counts.values()]),
        "postselected_P1": n_one / (n_zero + n_one) if n_zero + n_one > 0 else float("nan"),
    }


def expected_bias(n_bits):
    # Expected bias when n_bits fair bits are measured (normal approximation)
    return math.sqrt(2 / math.pi) / (2 * math.sqrt(n_bits))


def majority_error_prob(e):
    # Probability of misreading 3 or more of the 5
    return sum(math.comb(N_QUBITS, k) * e ** k * (1 - e) ** (N_QUBITS - k)
               for k in range(MAJORITY, N_QUBITS + 1))


def readout_non_ghz(e):
    # Everything except (1-e)^5 (read correctly) and e^5 (all 5 flip to the other GHZ outcome)
    return 1 - (1 - e) ** N_QUBITS - e ** N_QUBITS


print("decode_c3('01000') =", decode_c3("01000"), "(one 1, so corrected to 0)")
print("decode_c3('11011') =", decode_c3("11011"), "(four 1s, so corrected to 1)")
print(f"M(0.03)               = {majority_error_prob(0.03):.3e}")
print(f"readout_non_ghz(0.03) = {readout_non_ghz(0.03):.5f}")

<!-- 次に、評価結果を表で表示する関数を定義します。 -->

Next, we define functions that display the evaluation results as tables.

<!--
- `show_weights(r)`：(2) 1 の個数ごとの shot 数と割合、多数決の結果の表
- `show_table(rows, first_col)`：複数の評価結果を 1 つの表に並べる（4.5 以降で使う）
- `show_metrics(r)`：多数決で得たビットの P(0)・P(1)・bias・エントロピーと、公平なビットの目安を表示する（4.4・5.6 で使う）
-->

- `show_weights(r)`: table of (2) the number and fraction of shots for each number of 1s, and the result of the majority vote
- `show_table(rows, first_col)`: lists multiple evaluation results in one table (used from 4.5 onward)
- `show_metrics(r)`: displays P(0), P(1), bias, and entropies of the bits from the majority vote, with reference values for fair bits (used in 4.4 and 5.6)

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `show_weights` | 1 の個数ごとの表を表示。戻り値は `None` | `show_weights(evaluate_c3({"00000": 5, "11111": 4, "01000": 1}))` |
| `show_table` | Markdown の表を表示。戻り値は `None` | `show_table([("H", evaluate_c3({"00000": 5, "11111": 5}))], "gate")` |
| `show_metrics` | 多数決で得たビットの指標を表示。戻り値は `None` | `show_metrics(evaluate_c3({"00000": 5, "11111": 5}))` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `show_weights` | Displays the table by number of 1s. Returns `None` | `show_weights(evaluate_c3({"00000": 5, "11111": 4, "01000": 1}))` |
| `show_table` | Displays a Markdown table. Returns `None` | `show_table([("H", evaluate_c3({"00000": 5, "11111": 5}))], "gate")` |
| `show_metrics` | Displays the metrics of the bits from the majority vote. Returns `None` | `show_metrics(evaluate_c3({"00000": 5, "11111": 5}))` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>show_weights</code></td><td><code>r</code></td><td><code>evaluate_c3</code> の戻り値</td><td>必須</td></tr>
<tr><td rowspan="2"><code>show_table</code></td><td><code>rows</code></td><td><code>(ラベル, evaluate_c3 の戻り値)</code> のリスト。例：<code>[("H", r)]</code></td><td>必須</td></tr>
<tr><td><code>first_col</code></td><td>表の先頭列の見出し文字列。例：<code>"gate"</code>、<code>"trial"</code></td><td>必須</td></tr>
<tr><td><code>show_metrics</code></td><td><code>r</code></td><td><code>evaluate_c3</code> の戻り値</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>show_weights</code></td><td><code>r</code></td><td>Return value of <code>evaluate_c3</code></td><td>Required</td></tr>
<tr><td rowspan="2"><code>show_table</code></td><td><code>rows</code></td><td>List of <code>(label, return value of evaluate_c3)</code>. E.g., <code>[("H", r)]</code></td><td>Required</td></tr>
<tr><td><code>first_col</code></td><td>Heading string for the first column of the table. E.g., <code>"gate"</code>, <code>"trial"</code></td><td>Required</td></tr>
<tr><td><code>show_metrics</code></td><td><code>r</code></td><td>Return value of <code>evaluate_c3</code></td><td>Required</td></tr>
</tbody>
</table>


In [ ]:
def show_weights(r):
    lines = ["| number of 1s | counts | fraction | decoded bit | note |", "|---|---|---|---|---|"]
    for k, n in enumerate(r["weight_counts"]):
        if k in (0, N_QUBITS):
            note = "GHZ"
        else:
            note = "corrected by majority vote"
        decoded = "1" if k >= MAJORITY else "0"
        lines.append(f"| {k} | {n} | {n / r['shots']:.5f} | {decoded} | {note} |")
    display(Markdown("\n".join(lines)))


def show_table(rows, first_col):
    # rows: list of (label, result of evaluate_c3)
    lines = [
        f"| {first_col} | counts (0) | counts (1) | P(1) | bias | Shannon entropy | min-entropy | non-GHZ fraction |",
        "|---|---|---|---|---|---|---|---|",
    ]
    for label, r in rows:
        lines.append(
            f"| {label} | {r['n0']} | {r['n1']} | {r['P1']:.5f} | {r['bias']:.5f} "
            f"| {r['shannon']:.5f} | {r['min_entropy']:.5f} | {r['non_ghz']:.5f} |"
        )
    display(Markdown("\n".join(lines)))
    print("counts, P(1), bias, and entropies are values for the bits from the majority vote.")


def show_metrics(r):
    # Metrics of the bits from the majority vote, with reference values for fair bits
    for label, key in [("P(0)", "P0"), ("P(1)", "P1"), ("bias", "bias"),
                       ("Shannon entropy", "shannon"), ("min-entropy", "min_entropy")]:
        print(f"{label:15} = {r[key]:.6f}")
    b_ref = expected_bias(r["shots"])
    print(f"[Reference] expected bias for fair bits = {b_ref:.6f}")
    print(f"[Reference] min-entropy at that bias    = {-math.log2(1 / 2 + b_ref):.6f}")

<!-- ## 3. C3 回路を作る -->

## 3. Building the C3 circuit

<!-- 説明パート A・B の C3 回路を作る関数 `build_c3(gate)` を定義します。 -->

We define the function `build_c3(gate)`, which builds the C3 circuit of Explanation Parts A and B.

<!--
1. `QuantumCircuit(5, 5)` で量子ビット 5 個・古典ビット 5 個の回路を作る（量子ビットはすべて $|0\rangle$ から始まる）
2. 量子ビット 2（`CONTROL`）に、選んだゲート（$H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ のいずれか）を作用させる
3. 量子ビット 2 を制御ビットとして、量子ビット 1, 3, 0, 4（`TARGETS`）の順に CX ゲートをかける
4. 量子ビット 0〜4 を測定し、結果を古典ビット 0〜4 に記録する
-->

1. Create a circuit with 5 qubits and 5 classical bits using `QuantumCircuit(5, 5)` (all qubits start in $|0\rangle$)
2. Apply the chosen gate (one of $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$) to qubit 2 (`CONTROL`)
3. With qubit 2 as the control, apply CX gates to qubits 1, 3, 0, and 4 (`TARGETS`) in that order
4. Measure qubits 0–4 and record the results in classical bits 0–4

<!-- 3 つのゲートすべてについて、次のことを確かめます。 -->

For all three gates, we check the following.

<!--
- 元の回路と transpile 後の回路、ゲートの個数（説明パート D の表と比べる）
- transpile 後の CZ ゲートがすべて量子ビット 2 を含むこと（星形の接続グラフで実行できること）
- `Operator(...).equiv(...)` で、変換の前後で回路が同じ演算になっていること
- `Statevector` で、測定前の状態が GHZ 状態（$|00000\rangle$ と $|11111\rangle$ が確率 1/2 ずつ）になっていること
-->

- The original circuit, the transpiled circuit, and the gate counts (compare with the table in Explanation Part D)
- That all CZ gates after transpilation include qubit 2 (so that the circuit can run on the star-shaped connectivity graph)
- That the circuit implements the same operation before and after the conversion, using `Operator(...).equiv(...)`
- That the state before measurement is the GHZ state ($|00000\rangle$ and $|11111\rangle$ each with probability 1/2), using `Statevector`

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `build_c3` | 測定を含む 5 量子ビット回路（QuantumCircuit） | `build_c3("H")` |
| `cz_pairs` | 回路の CZ ゲートが作用する量子ビットの組のリスト（例：`[(1, 2), ...]`） | `cz_pairs(transpile(build_c3("H"), basis_gates=BASIS_GATES, optimization_level=0))` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `build_c3` | 5-qubit circuit including measurement (QuantumCircuit) | `build_c3("H")` |
| `cz_pairs` | List of the qubit pairs on which the circuit's CZ gates act (e.g., `[(1, 2), ...]`) | `cz_pairs(transpile(build_c3("H"), basis_gates=BASIS_GATES, optimization_level=0))` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>build_c3</code></td><td><code>gate</code></td><td>大文字小文字を区別して <code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>cz_pairs</code></td><td><code>qc</code></td><td>transpile 後の回路（QuantumCircuit）</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>build_c3</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code> (case-sensitive)</td><td>Required</td></tr>
<tr><td><code>cz_pairs</code></td><td><code>qc</code></td><td>Transpiled circuit (QuantumCircuit)</td><td>Required</td></tr>
</tbody>
</table>


In [ ]:
def build_c3(gate):
    if gate not in GATES:
        raise ValueError(f"gate must be one of {GATES} (got: {gate!r})")

    qc = QuantumCircuit(N_QUBITS, N_QUBITS)

    # 1. Create a superposition on the central qubit
    if gate == "H":
        qc.h(CONTROL)
    elif gate == "Rx":
        qc.rx(math.pi / 2, CONTROL)
    else:  # "Ry"
        qc.ry(math.pi / 2, CONTROL)

    # 2. Copy the value with CX gates to form the GHZ state (order of Fig. 4 of the paper)
    for target in TARGETS:
        qc.cx(CONTROL, target)

    # 3. Measure all qubits
    qc.measure(range(N_QUBITS), range(N_QUBITS))
    return qc


def cz_pairs(qc):
    # Qubit pairs acted on by the CZ gates
    return [tuple(sorted(qc.find_bit(q).index for q in inst.qubits))
            for inst in qc.data if inst.operation.name == "cz"]


for gate in GATES:
    qc = build_c3(gate)
    qc_t = transpile(qc, basis_gates=BASIS_GATES, optimization_level=0)

    print(f"===== {gate} =====")
    print("[Original circuit]")
    print(qc.draw())
    print("[Transpiled circuit]")
    print(qc_t.draw(fold=-1))
    print("Gate counts:", dict(qc_t.count_ops()))

    pairs = cz_pairs(qc_t)
    print("Qubit pairs of the CZ gates:", pairs)
    print("All CZ gates include qubit 2:", all(CONTROL in p for p in pairs))

    # Check, excluding measurements, that the operation is the same before and after conversion
    qc_u = qc.remove_final_measurements(inplace=False)
    same = Operator(qc_u).equiv(Operator(qc_t.remove_final_measurements(inplace=False)))
    print("Same operation before and after conversion:", same)

    # Statevector before measurement
    sv = Statevector(qc_u)
    print("State probabilities before measurement:", {str(x): round(float(p), 6) for x, p in sv.probabilities_dict().items()})
    print("Nonzero amplitudes:", {str(x): complex(np.round(a, 6)) for x, a in sv.to_dict().items()})
    print()

<!-- ## 4. ノイズなしのシミュレーション -->

## 4. Noiseless simulation

<!-- まずノイズのない理想的なシミュレータで C3 回路を測定します。理想的には $|00000\rangle$ と $|11111\rangle$ だけが現れ、多数決で得たビットは公平な 1 ビットになります。 -->

First, we measure the C3 circuit on an ideal simulator with no noise. Ideally only $|00000\rangle$ and $|11111\rangle$ appear, and the bit from the majority vote is a fair bit.

<!-- ### 4.1 ゲートと測定回数を選ぶ -->

### 4.1 Choosing the gate and the number of measurements

<!-- `GATE_IDEAL` に `"H"`, `"Rx"`, `"Ry"` のいずれかを、`SHOTS_IDEAL` に測定回数（shot 数）を指定します。多数決で得られる乱数のビット数は `SHOTS_IDEAL` と同じです。 -->

Set `GATE_IDEAL` to one of `"H"`, `"Rx"`, `"Ry"`, and set `SHOTS_IDEAL` to the number of measurements (shots). The number of random bits obtained by majority vote is the same as `SHOTS_IDEAL`.

In [ ]:
GATE_IDEAL = "H"          # Choose from "H", "Rx", "Ry"
SHOTS_IDEAL = 1_000_000   # Number of measurements (= number of random bits)

<!-- ### 4.2 回路を作って transpile する -->

### 4.2 Building and transpiling the circuit

<!-- `build_c3` で回路を作り、`transpile` でネイティブゲート `BASIS_GATES` だけを使う回路に変換します（`optimization_level=0`）。 -->

We build the circuit with `build_c3` and use `transpile` to convert it into a circuit that uses only the native gates `BASIS_GATES` (`optimization_level=0`).

In [ ]:
qc_ideal = build_c3(GATE_IDEAL)
qc_ideal_t = transpile(qc_ideal, basis_gates=BASIS_GATES, optimization_level=0)

print(qc_ideal_t.draw(fold=-1))

<!-- ### 4.3 測定する -->

### 4.3 Measuring

<!-- `AerSimulator()` をノイズなしで作り、`backend.run(..., shots=SHOTS_IDEAL, memory=True)` で測定します。 -->

We create a noiseless `AerSimulator()` and measure with `backend.run(..., shots=SHOTS_IDEAL, memory=True)`.

<!--
- `result.get_counts()`：5 ビットの結果ごとの回数
- `result.get_memory()`：各 shot の 5 ビットの文字列を測定順に並べたリスト（生のデータ）。シミュレータの結果は古典コンピュータの擬似乱数から作られ、シードで再現できるので、真の量子乱数ではなく、暗号には使えません。
-->

- `result.get_counts()`: the count for each 5-bit outcome
- `result.get_memory()`: a list of the 5-bit string of each shot in measurement order (the raw data). Simulator results are generated from a classical computer's pseudorandom numbers and are reproducible with a seed, so they are not true quantum random numbers and must not be used for cryptography.

<!-- 各 shot の文字列に `decode_c3` を適用して多数決をとり、乱数のビット列にします。最初の 10 shot について、文字列・1 の個数・多数決の結果を並べて表示します。 -->

We apply `decode_c3` to the string of each shot to take the majority vote and form the random bit string. For the first 10 shots, we list the string, the number of 1s, and the result of the majority vote.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、このセルの raw dataを `4.3_ideal_single/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data of this cell is saved in `4.3_ideal_single/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
backend_ideal = AerSimulator(seed_simulator=SEED)

job = backend_ideal.run(qc_ideal_t, shots=SHOTS_IDEAL, memory=True)
result = job.result()

counts_ideal = result.get_counts()
raw_shots_ideal = result.get_memory()
decoded_bits_ideal = "".join(decode_c3(s) for s in raw_shots_ideal)

print("Counts for each 5-bit outcome =", counts_ideal)
print()
print("shot | string |  # of 1s | majority")
for i, s in enumerate(raw_shots_ideal[:10]):
    print(f"{i:4d} | {s}  | {hamming_weight(s):8d} | {decode_c3(s)}")

print()
print("Bit string from the majority vote (first 50 bits) =", decoded_bits_ideal[:50])
print("Length of the bit string from the majority vote =", len(decoded_bits_ideal))

# Save the raw data (only when SAVE_RESULTS = True)
if SAVE_RESULTS:
    path = save_raw(output_dir("4.3_ideal_single") / f"c3_{GATE_IDEAL}_ideal.txt", raw_shots_ideal,
                    GATE_IDEAL, SHOTS_IDEAL, SEED, metadata={"condition": "ideal"})
    print("Saved:", path)

<!-- ### 4.4 評価指標を計算する -->

### 4.4 Computing the evaluation metrics

<!-- `evaluate_c3` で指標を計算します。 -->

`evaluate_c3` computes the metrics.

<!--
1. **1 の個数の分布**：ノイズがないので、1 の個数は 0 か 5 だけのはずです（non-GHZ の割合は 0）。
2. **多数決で得たビット**：C1 と同じ指標です。bias を揺らぎの目安 $\sqrt{2/\pi}\,/\,(2\sqrt{N})$ と比べます。
3. **5 ビットの結果の min-entropy**：理想的には約 1 ビットです（説明パート C の (3)）。5 ビットの結果でも、乱数は 1 ビット分しかないことを確かめてください。
-->

1. **Distribution of the number of 1s**: Since there is no noise, the number of 1s should be only 0 or 5 (the non-GHZ fraction is 0).
2. **Bit from the majority vote**: the same metrics as C1. Compare the bias with the reference fluctuation $\sqrt{2/\pi}\,/\,(2\sqrt{N})$.
3. **Min-entropy of the 5-bit outcomes**: ideally about 1 bit (item (3) of Explanation Part C). Confirm that even the 5-bit outcome holds only 1 bit of randomness.

In [ ]:
r = evaluate_c3(counts_ideal)

print("===== Distribution of the number of 1s =====")
show_weights(r)
print(f"non-GHZ fraction = {r['non_ghz']:.6f}")

print()
print("===== Bits from the majority vote =====")
show_metrics(r)

print()
print("===== 5-bit outcomes =====")
print(f"min-entropy (5-bit distribution) = {r['raw_min_entropy']:.6f} (ideal: 1)")

<!-- ### 4.5 測定を繰り返す -->

### 4.5 Repeating the measurement

<!-- 同じ測定を `N_TRIALS` 回繰り返し、ばらつきを見ます。以降でも使うので、「回路を作る → transpile → 測定 → 評価」を関数 `run_c3` にまとめます。評価に必要な量はすべて `counts` から計算できるので、生のデータを保存しないときは `memory=True` を付けません。 -->

We repeat the same measurement `N_TRIALS` times and look at the spread. Because it is also used later, we package "build the circuit → transpile → measure → evaluate" into the function `run_c3`. All quantities needed for evaluation can be computed from `counts`, so `memory=True` is not set when raw data is not being saved.

<!-- `run_c3` に `raw_path` を指定すると、生のデータを別ファイルに保存します。 -->

If `raw_path` is passed to `run_c3`, the raw data is saved to separate files.

```python
r = run_c3("H", 1_000_000, seed=2026, raw_path="results/c3/manual/c3_H_ideal.txt")
```

<!--
| ファイル | 内容 |
|---|---|
| `c3_H_ideal.txt` | 多数決で得たビットを shot 順に並べた 0/1 の列（改行なし、長さ $N$）。SP 800-90B や抽出器に渡すのはこのファイル |
| `c3_H_ideal.shots.txt` | 各 shot の 5 ビットの文字列（Qiskit の順：量子ビット 4, 3, 2, 1, 0）を 1 行に 1 つずつ並べたもの |
| `c3_H_ideal.txt.json` | 測定条件・ノイズモデル・counts・評価結果 |
-->

| File | Contents |
|---|---|
| `c3_H_ideal.txt` | Sequence of 0s and 1s (no line breaks, length $N$) of the bits from the majority vote in shot order. This is the file passed to SP 800-90B and the extractors |
| `c3_H_ideal.shots.txt` | The 5-bit string of each shot (Qiskit order: qubits 4, 3, 2, 1, 0), one per line |
| `c3_H_ideal.txt.json` | Measurement conditions, noise model, counts, and evaluation results |

<!--
既存ファイルがある場合は上書きせずエラーにします。保存先を省略すると、集計結果だけを返します。
繰り返し測定は `run_trials(..., raw_dir="results/c3/manual/trials")` で試行ごとに保存できます。
-->

If a file already exists, an error is raised rather than overwriting it. If the destination is omitted, only the aggregated results are returned.
For repeated measurements, `run_trials(..., raw_dir="results/c3/manual/trials")` saves the data for each trial.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、次のセルの最後の `run_trials` に `raw_dir` として `RUN_DIR/4.5_ideal_trials` を渡すので、試行ごとのファイル（`c3_H_ideal_trial001.txt` など）と、全試行の一覧 `trials.csv` を保存します。上の例のように自分で `raw_path` や `raw_dir` を指定した場合は、`SAVE_RESULTS` に関係なくその場所に保存します。 -->

If `SAVE_RESULTS = True` in 1., the `run_trials` call at the end of the next cell is given `RUN_DIR/4.5_ideal_trials` as `raw_dir`, so one file per trial (`c3_H_ideal_trial001.txt` and so on) and the list of all trials, `trials.csv`, are saved. If you give your own `raw_path` or `raw_dir` as in the examples above, the files are saved there regardless of `SAVE_RESULTS`.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `run_c3` | `evaluate_c3` と同じ評価結果の辞書。保存時も戻り値は同じ | `run_c3("H", 1_000_000, seed=2026, raw_path="results/c3/manual/c3_H_ideal.txt")` |
| `run_trials` | `(試行番号, 評価結果の辞書)` のリスト。`raw_dir` を指定すると、試行ごとのファイルと `trials.csv` を保存する | `run_trials("H", 1_000_000, 3, seed=2026, raw_dir="results/c3/manual/trials")` |
| `run_conditions` | `(ラベル, 評価結果の辞書)` のリスト。`(ゲート, 条件の名前, NoiseModel)` の組を 1 回ずつ測定する（4.6・5.8 で使う）。`SAVE_RESULTS = True` なら raw data と `summary.csv` を保存する | `run_conditions([("H", "ideal", None)], 1_000, "4.6_ideal_gates")` |
| `offset_seed` | `seed + k`（`seed` が `None` なら `None`）。測定ごとに違うシードにする | `offset_seed(2026, 3)` |
| `summarize` | bias・min-entropy・non-GHZ の割合の要約を表示。戻り値は `None` | `summarize(rows_ideal)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `run_c3` | Evaluation dictionary, same as `evaluate_c3`. The return value is the same when saving | `run_c3("H", 1_000_000, seed=2026, raw_path="results/c3/manual/c3_H_ideal.txt")` |
| `run_trials` | List of `(trial number, evaluation dictionary)`. If `raw_dir` is given, one file per trial and `trials.csv` are saved | `run_trials("H", 1_000_000, 3, seed=2026, raw_dir="results/c3/manual/trials")` |
| `run_conditions` | List of `(label, evaluation dictionary)`. Measures each `(gate, condition name, NoiseModel)` once (used in 4.6 and 5.8). With `SAVE_RESULTS = True`, the raw data and `summary.csv` are saved | `run_conditions([("H", "ideal", None)], 1_000, "4.6_ideal_gates")` |
| `offset_seed` | `seed + k` (`None` if `seed` is `None`). Gives each measurement a different seed | `offset_seed(2026, 3)` |
| `summarize` | Displays a summary of the bias, min-entropy, and non-GHZ fraction. Returns `None` | `summarize(rows_ideal)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="8"><code>run_c3</code></td><td><code>gate</code></td><td><code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>測定回数（shot 数）。1 以上の整数（例：<code>1_000_000</code>）</td><td>必須</td></tr>
<tr><td><code>noise_model</code></td><td><code>build_noise_model("both")</code> などで作った NoiseModel。ノイズなしなら <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>再現用の非負整数（例：<code>2026</code>）。毎回異なる乱数なら <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>raw_path</code></td><td>多数決で得たビット列の保存先の文字列または Path（例：<code>"results/c3/manual/c3_H_ideal.txt"</code>）。<code>None</code> なら保存しない。未使用のファイル名を指定</td><td><code>None</code></td></tr>
<tr><td><code>method</code></td><td>通常は <code>"automatic"</code>。密度行列方式を指定する場合は <code>"density_matrix"</code></td><td><code>'automatic'</code></td></tr>
<tr><td><code>optimization_level</code></td><td>Qiskit の回路変換時の最適化レベル。<code>0</code>・<code>1</code>・<code>2</code>・<code>3</code>、または Qiskit の既定値を使う <code>None</code>。このノートブックではゲートの個数を固定するため <code>0</code></td><td><code>0</code></td></tr>
<tr><td><code>metadata</code></td><td>JSON に保存できる追加情報の辞書（例：<code>{"trial": 1}</code>）。不要なら <code>None</code>。<code>raw_path</code> 指定時に保存</td><td><code>None</code></td></tr>
<tr><td rowspan="7"><code>run_trials</code></td><td><code>gate</code></td><td><code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>1 試行あたりの測定回数。1 以上の整数</td><td>必須</td></tr>
<tr><td><code>n_trials</code></td><td>繰り返す試行数。1 以上の整数（例：<code>3</code>）</td><td>必須</td></tr>
<tr><td><code>noise_model</code></td><td>NoiseModel。ノイズなしなら <code>None</code></td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>非負整数または <code>None</code>。整数の場合は試行番号を加えたシードで実行</td><td><code>None</code></td></tr>
<tr><td><code>raw_dir</code></td><td>試行別のファイルを保存するフォルダ。文字列または Path。<code>None</code> なら保存しない。既存ファイルは上書きしない</td><td><code>None</code></td></tr>
<tr><td><code>label</code></td><td>ファイル名に入れる条件の名前（例：<code>"ideal"</code>、<code>"noise-both"</code>）。<code>trials.csv</code> の <code>condition</code> 列にもなる</td><td><code>"ideal"</code></td></tr>
<tr><td rowspan="3"><code>run_conditions</code></td><td><code>conditions</code></td><td><code>(ゲート, 条件の名前, NoiseModel または None)</code> のリスト。条件の名前はファイル名に入る</td><td>必須</td></tr>
<tr><td><code>shots</code></td><td>1 回あたりの測定回数</td><td>必須</td></tr>
<tr><td><code>folder</code></td><td><code>RUN_DIR</code> の中の保存先のフォルダ名（例：<code>"4.6_ideal_gates"</code>）</td><td>必須</td></tr>
<tr><td rowspan="2"><code>offset_seed</code></td><td><code>seed</code></td><td>非負整数または <code>None</code></td><td>必須</td></tr>
<tr><td><code>k</code></td><td>足す数（試行番号や条件の番号）</td><td>必須</td></tr>
<tr><td><code>summarize</code></td><td><code>rows</code></td><td><code>run_trials</code> が返す <code>(試行番号, 評価結果)</code> のリスト。少なくとも 1 試行が必要</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="8"><code>run_c3</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements (shots). An integer of at least 1 (e.g., <code>1_000_000</code>)</td><td>Required</td></tr>
<tr><td><code>noise_model</code></td><td>A NoiseModel created with, e.g., <code>build_noise_model("both")</code>. <code>None</code> for no noise</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>Non-negative integer for reproducibility (e.g., <code>2026</code>). <code>None</code> for different random numbers each time</td><td><code>None</code></td></tr>
<tr><td><code>raw_path</code></td><td>Destination for the bit string from the majority vote, as a string or Path (e.g., <code>"results/c3/manual/c3_H_ideal.txt"</code>). <code>None</code> means do not save. Specify an unused file name</td><td><code>None</code></td></tr>
<tr><td><code>method</code></td><td>Normally <code>"automatic"</code>. Use <code>"density_matrix"</code> to select the density-matrix method</td><td><code>'automatic'</code></td></tr>
<tr><td><code>optimization_level</code></td><td>Optimization level for Qiskit's circuit conversion. <code>0</code>, <code>1</code>, <code>2</code>, <code>3</code>, or <code>None</code> to use Qiskit's default. This notebook uses <code>0</code> to fix the number of gates</td><td><code>0</code></td></tr>
<tr><td><code>metadata</code></td><td>Dictionary of extra information that can be saved as JSON (e.g., <code>{"trial": 1}</code>). <code>None</code> if not needed. Saved when <code>raw_path</code> is specified</td><td><code>None</code></td></tr>
<tr><td rowspan="7"><code>run_trials</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements per trial. An integer of at least 1</td><td>Required</td></tr>
<tr><td><code>n_trials</code></td><td>Number of trials to repeat. An integer of at least 1 (e.g., <code>3</code>)</td><td>Required</td></tr>
<tr><td><code>noise_model</code></td><td>A NoiseModel. <code>None</code> for no noise</td><td><code>None</code></td></tr>
<tr><td><code>seed</code></td><td>Non-negative integer or <code>None</code>. If an integer, each trial runs with the seed plus the trial number</td><td><code>None</code></td></tr>
<tr><td><code>raw_dir</code></td><td>Folder in which to save the files of each trial, as a string or Path. <code>None</code> means do not save. Existing files are not overwritten</td><td><code>None</code></td></tr>
<tr><td><code>label</code></td><td>Name of the condition put in the file names (e.g., <code>"ideal"</code>, <code>"noise-both"</code>). Also used as the <code>condition</code> column of <code>trials.csv</code></td><td><code>"ideal"</code></td></tr>
<tr><td rowspan="3"><code>run_conditions</code></td><td><code>conditions</code></td><td>List of <code>(gate, condition name, NoiseModel or None)</code>. The condition name goes into the file name</td><td>Required</td></tr>
<tr><td><code>shots</code></td><td>Number of measurements per condition</td><td>Required</td></tr>
<tr><td><code>folder</code></td><td>Name of the destination folder in <code>RUN_DIR</code> (e.g., <code>"4.6_ideal_gates"</code>)</td><td>Required</td></tr>
<tr><td rowspan="2"><code>offset_seed</code></td><td><code>seed</code></td><td>Non-negative integer or <code>None</code></td><td>Required</td></tr>
<tr><td><code>k</code></td><td>Number to add (trial number or condition number)</td><td>Required</td></tr>
<tr><td><code>summarize</code></td><td><code>rows</code></td><td>List of <code>(trial number, evaluation result)</code> returned by <code>run_trials</code>. At least one trial is required</td><td>Required</td></tr>
</tbody>
</table>

<br>

<!-- `run_c3` の `method`・`optimization_level`・`metadata` は、`method="density_matrix"` のように引数名を付けて指定します。C1・C2 の `run_c1`・`run_c2` と違い、`optimization_level` の既定値は `0` です（説明パート D）。 -->

Specify `method`, `optimization_level`, and `metadata` of `run_c3` by keyword, as in `method="density_matrix"`. Unlike `run_c1` and `run_c2` for C1 and C2, the default of `optimization_level` is `0` (Explanation Part D).

In [ ]:
N_TRIALS = 10


def offset_seed(seed, k):
    # A different seed for each run: seed + k (None stays None, i.e., different results each time)
    return None if seed is None else seed + k


def run_c3(gate, shots, noise_model=None, seed=None, raw_path=None,
           *, method="automatic", optimization_level=0, metadata=None):
    # Build the circuit -> transpile -> measure -> evaluate. With raw_path, the raw data is also saved.
    if raw_path is not None:
        ensure_new(raw_path)   # stop before measuring if the files already exist
    qc_t = transpile(build_c3(gate), basis_gates=BASIS_GATES, optimization_level=optimization_level)
    backend = AerSimulator(method=method, noise_model=noise_model, seed_simulator=seed)
    result = backend.run(qc_t, shots=shots, memory=raw_path is not None).result()
    if raw_path is not None:
        save_raw(raw_path, result.get_memory(), gate, shots, seed, noise_model,
                 method=method, optimization_level=optimization_level, metadata=metadata)
    return evaluate_c3(result.get_counts())


def run_trials(gate, shots, n_trials, noise_model=None, seed=None, raw_dir=None, label="ideal"):
    # Repeat the same measurement. With raw_dir, one file per trial and trials.csv are saved.
    rows, records = [], []
    for trial in range(1, n_trials + 1):
        name = f"c3_{gate}_{label}_trial{trial:03d}.txt"
        raw_path = None if raw_dir is None else Path(raw_dir) / name
        s = offset_seed(seed, trial)
        r = run_c3(gate, shots, noise_model, s, raw_path, metadata={"condition": label, "trial": trial})
        rows.append((trial, r))
        records.append(dict(file=name, gate=gate, condition=label, trial=trial, seed=s, **r))
    if raw_dir is not None:
        write_csv(Path(raw_dir) / "trials.csv", records)   # written after all trials have finished
    return rows


def run_conditions(conditions, shots, folder):
    # Measure each (gate, condition, noise_model) once (used in 4.6 and 5.8).
    # With SAVE_RESULTS = True, the raw data and summary.csv are saved in output_dir(folder).
    out = output_dir(folder)
    rows, records = [], []
    for k, (gate, condition, noise_model) in enumerate(conditions):
        name = f"c3_{gate}_{condition}.txt"
        raw_path = None if out is None else out / name
        s = offset_seed(SEED, k)   # the same seed would align the results of different conditions
        r = run_c3(gate, shots, noise_model, s, raw_path, metadata={"condition": condition})
        rows.append((f"{gate} / {condition}", r))
        records.append(dict(file=name, gate=gate, condition=condition, seed=s, **r))
    if out is not None:
        write_csv(out / "summary.csv", records)
    return rows


def summarize(rows):
    mean = lambda xs: sum(xs) / len(xs)
    biases = [r["bias"] for _, r in rows]
    mins = [r["min_entropy"] for _, r in rows]
    non_ghz = [r["non_ghz"] for _, r in rows]
    print(f"mean bias             = {mean(biases):.6f} (reference {expected_bias(rows[0][1]['shots']):.6f})")
    print(f"mean min-entropy      = {mean(mins):.6f} (min {min(mins):.6f}, max {max(mins):.6f})")
    print(f"mean non-GHZ fraction = {mean(non_ghz):.5f}")


rows_ideal = run_trials(GATE_IDEAL, SHOTS_IDEAL, N_TRIALS, seed=SEED, raw_dir=output_dir("4.5_ideal_trials"))
show_table(rows_ideal, "trial")
summarize(rows_ideal)

<!-- ### 4.6 （参考）3 つのゲートを比べる -->

### 4.6 (Optional) Comparing the three gates

<!-- 同じ測定回数で $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, $R_Y(\pi/2)$ を 1 回ずつ測定して並べます。ノイズがないので、3 つの違いは統計的な揺らぎだけのはずです。 -->

We measure $H_{\mathrm{Gate}}$, $R_X(\pi/2)$, and $R_Y(\pi/2)$ once each with the same number of measurements and list the results side by side. Since there is no noise, the only differences among the three should be statistical fluctuations.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、3 つのゲートの raw data と一覧 `summary.csv`を `4.6_ideal_gates/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data for the three gates and their list, `summary.csv`, are saved in `4.6_ideal_gates/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
rows = run_conditions([(gate, "ideal", None) for gate in GATES], SHOTS_IDEAL, "4.6_ideal_gates")
show_table(rows, "gate / condition")

<!-- ## 5. ノイズありのシミュレーション -->

## 5. Noisy simulation

<!-- 次に、説明パート E のノイズを加えて測定します。 -->

Next, we add the noise of Explanation Part E and measure.

<!-- ### 5.1 ノイズのパラメータ -->

### 5.1 Noise parameters

<!-- 1 量子ビットゲートのパラメータは C1 と同じです。CZ ゲートの脱分極パラメータ $\lambda_{\mathrm{CZ}}$ とゲート時間を加えます。熱緩和誤差の関数には $T_1$, $T_2$, ゲート時間を**同じ単位**で渡す必要があるので、すべてナノ秒（ns）にそろえます。 -->

The 1-qubit gate parameters are the same as in C1. We add the CZ gate depolarizing parameter $\lambda_{\mathrm{CZ}}$ and gate time. The thermal relaxation error function requires $T_1$, $T_2$, and the gate time in the **same unit**, so everything is expressed in nanoseconds (ns).

<!-- 最後に `average_gate_fidelity` で、熱緩和と合わせた平均ゲート忠実度が 1 量子ビットゲートで 99.9%、CZ ゲートで 99.0% になっていることを確かめます（説明パート E）。 -->

Finally, we use `average_gate_fidelity` to confirm that the average gate fidelity including thermal relaxation is 99.9% for 1-qubit gates and 99.0% for CZ gates (Explanation Part E).

In [ ]:
DEPOLARIZING_PARAM = 0.00198      # Depolarizing parameter λ of 1-qubit gates (same as C1)
CZ_DEPOLARIZING_PARAM = 0.013245  # Depolarizing parameter λ_CZ of CZ gates (fidelity 99.0% including thermal relaxation)
READOUT_ERROR_RATE = 0.03         # Readout error (fidelity 97%)

T1 = 0.964e6                      # Relaxation time 0.964 ms = 0.964e6 ns
T2 = 1.155e6                      # Dephasing time 1.155 ms = 1.155e6 ns
GATE_TIME = 20                    # 1-qubit gate time 20 ns
CZ_GATE_TIME = 60                 # CZ gate time 60 ns

# Check the average gate fidelities
one_qubit_error = depolarizing_error(DEPOLARIZING_PARAM, 1).compose(
    thermal_relaxation_error(T1, T2, GATE_TIME))
cz_thermal = thermal_relaxation_error(T1, T2, CZ_GATE_TIME)
cz_thermal_pair = cz_thermal.tensor(cz_thermal)
cz_error = depolarizing_error(CZ_DEPOLARIZING_PARAM, 2).compose(cz_thermal_pair)

print(f"1-qubit gate: average gate fidelity = {average_gate_fidelity(one_qubit_error):.5f} (target 0.999)")
print(f"Average gate infidelity of CZ thermal relaxation alone = {1 - average_gate_fidelity(cz_thermal_pair):.4e}")
print(f"CZ gate     : average gate fidelity = {average_gate_fidelity(cz_error):.5f} (target 0.990)")

<!-- ### 5.2 ノイズモデルを作る関数 -->

### 5.2 Function for building the noise model

<!-- ノイズモデルは、ノイズの**成分**を組み合わせて作ります。次のセルの 3 つの部分でできています。 -->

The noise model is built by combining noise **components**. The next cell has three parts.

<!--
- `NOISE_COMPONENTS`：ノイズの成分の一覧。成分ごとに、強さから Qiskit Aer の誤差を作る関数を持ちます。ゲートの後に加える量子的な誤差は `gate_errors`、測定の読み出し誤差は `readout` に書きます。`gate_errors` のキーはゲートの名前（`"rx"`, `"ry"`, `"cz"`）です。
- `NOISE_KINDS`：5.3〜5.8 で使うノイズの種類。名前と、`{成分: 強さ}` の組です。`"gate"` は 1 量子ビットゲートと CZ ゲートの脱分極 + 熱緩和、`"readout"` は対称な読み出し誤差、`"both"` はその両方で、強さは 5.1 の定数です。
- `build_noise_model(kind)`：`NOISE_KINDS` の名前、または `{成分: 強さ}` の辞書から `NoiseModel` を作ります。同じゲートの誤差は書いた順に `compose` で合成し（脱分極のあとに熱緩和）、読み出し誤差は混同行列を掛け合わせて、それぞれ 1 回だけ加えます。Qiskit Aer では、読み出し誤差を 2 回加えると後のもので上書きされるためです。5.9 の `build_sweep_noise` も同じ関数を使います。
-->

- `NOISE_COMPONENTS`: the list of noise components. Each component has functions that create Qiskit Aer errors from a strength. Quantum errors added after gates go in `gate_errors`, and readout errors at measurement go in `readout`. The keys of `gate_errors` are gate names (`"rx"`, `"ry"`, `"cz"`).
- `NOISE_KINDS`: the noise kinds used in 5.3–5.8. Each is a name and a set of `{component: strength}`. `"gate"` is depolarizing + thermal relaxation on 1-qubit gates and CZ gates, `"readout"` is a symmetric readout error, and `"both"` is both; the strengths are the constants in 5.1.
- `build_noise_model(kind)`: builds a `NoiseModel` from a name in `NOISE_KINDS` or from a dictionary `{component: strength}`. Errors on the same gate are composed with `compose` in the order written (depolarizing, then thermal relaxation), readout errors are combined by multiplying their confusion matrices, and each is added only once, because in Qiskit Aer adding a second readout error replaces the first. `build_sweep_noise` in 5.9 uses the same function.

<!-- このノートブックで定義している成分は次のとおりです。 -->

The components defined in this notebook are as follows.

<!--
| 成分 | 加える誤差 | 強さ |
|---|---|---|
| `depolarizing` | `rx`, `ry` の後に `depolarizing_error(λ, 1)` | λ（0〜1） |
| `cz_depolarizing` | `cz` の後に `depolarizing_error(λ_CZ, 2)` | $\lambda_{\mathrm{CZ}}$（0〜1） |
| `thermal` | `rx`, `ry` の後に `thermal_relaxation_error(T1, T2, 倍率 × GATE_TIME)`、`cz` の後にその 2 量子ビット版（`tensor` で並べる。時間は `倍率 × CZ_GATE_TIME`） | ゲート時間の倍率（1 が 5.1 の値） |
| `readout` | 対称な読み出し誤差の混同行列 | ε（0〜1） |
| `readout_asym` | $P(1\mid 0) = 0$, $P(0\mid 1) = \delta$ の混同行列 | δ（0〜1） |
-->

| Component | Error added | Strength |
|---|---|---|
| `depolarizing` | `depolarizing_error(λ, 1)` after `rx`, `ry` | λ (0–1) |
| `cz_depolarizing` | `depolarizing_error(λ_CZ, 2)` after `cz` | $\lambda_{\mathrm{CZ}}$ (0–1) |
| `thermal` | `thermal_relaxation_error(T1, T2, multiplier × GATE_TIME)` after `rx`, `ry`, and its 2-qubit version after `cz` (placed side by side with `tensor`; time `multiplier × CZ_GATE_TIME`) | Gate-time multiplier (1 is the value in 5.1) |
| `readout` | Confusion matrix of a symmetric readout error | ε (0–1) |
| `readout_asym` | Confusion matrix with $P(1\mid 0) = 0$, $P(0\mid 1) = \delta$ | δ (0–1) |

<!-- **ノイズの種類を増やす手順** -->

**How to add a kind of noise**

<!--
1. `NOISE_COMPONENTS` に成分を 1 つ加えます（下の例と、その下の Qiskit Aer の誤差の表を参照）。
2. 5.3〜5.8 で使うときは、`NOISE_KINDS` に名前と `{成分: 強さ}` を加え、5.3 の `NOISE_KIND` にその名前を指定します。5.8 の比較と、保存するファイルの名前（`noise-<名前>`）は自動で増えます。
3. 5.9 で強さを変えるときは、5.9 の `SWEEP_VALUES` に成分の名前と強さのリストを加えます。グラフの列、CSV、保存のフォルダは自動で増えます。
-->

1. Add one component to `NOISE_COMPONENTS` (see the example below and the table of Qiskit Aer errors after it).
2. To use it in 5.3–5.8, add a name and `{component: strength}` to `NOISE_KINDS`, and set `NOISE_KIND` in 5.3 to that name. The comparison in 5.8 and the names of the saved files (`noise-<name>`) follow automatically.
3. To vary its strength in 5.9, add the component name and a list of strengths to `SWEEP_VALUES` in 5.9. The plot columns, the CSV files, and the saved folders follow automatically.

<!-- 例：エネルギー緩和だけを表す振幅減衰を加える場合（次のセルの後に、新しいセルで実行します） -->

Example: adding amplitude damping, which models energy relaxation only (run in a new cell after the next cell)

```python
from qiskit_aer.noise import amplitude_damping_error

NOISE_COMPONENTS["amplitude_damping"] = dict(
    label="Amplitude damping probability gamma", max=1,
    gate_errors=lambda s: {("rx", "ry"): amplitude_damping_error(s)})
NOISE_KINDS["amp_damping"] = {"amplitude_damping": 0.01}   # NOISE_KIND = "amp_damping" in 5.3
# In the settings cell of 5.9: SWEEP_VALUES["amplitude_damping"] = [i / 50 for i in range(11)]
```

<!-- **Qiskit Aer で加えられるノイズ**（qiskit-aer 0.17） -->

**Noise that Qiskit Aer can add** (qiskit-aer 0.17)

<!-- 量子的な誤差（`QuantumError`）はゲートの後に、古典的な読み出し誤差（`ReadoutError`）は測定のときに加わります。量子的な誤差は `compose`（順に作用）や `tensor`（量子ビットを並べる）で組み合わせられます。 -->

Quantum errors (`QuantumError`) act after gates, and classical readout errors (`ReadoutError`) act at measurement. Quantum errors can be combined with `compose` (applied in sequence) and `tensor` (placed on qubits side by side).

<!--
| 関数（`qiskit_aer.noise`） | 種類 | 主な引数 | 内容・使いどころ |
|---|---|---|---|
| `depolarizing_error(param, num_qubits)` | 量子（ゲートの後） | 脱分極パラメータ λ、量子ビット数 | 確率 λ で状態を最大混合状態に置き換える。ゲートの不忠実度をまとめて表す標準的なモデル |
| `thermal_relaxation_error(t1, t2, time, excited_state_population=0)` | 量子 | $T_1$, $T_2$, 時間 | エネルギー緩和（$\|1\rangle \to \|0\rangle$）と位相緩和。ゲート時間や待ち時間の間の緩和 |
| `amplitude_damping_error(param_amp, excited_state_population=0)` | 量子 | 減衰確率 γ | エネルギー緩和だけ（$\|1\rangle \to \|0\rangle$） |
| `phase_damping_error(param_phase)` | 量子 | 位相減衰の強さ | 位相緩和だけ。重ね合わせの干渉が弱まるが、計算基底での測定確率は変えない |
| `phase_amplitude_damping_error(param_amp, param_phase, excited_state_population=0)` | 量子 | γ と位相減衰の強さ | 上の 2 つを同時に |
| `pauli_error([("X", p), ("I", 1 - p)])` | 量子 | パウリ演算子と確率の組 | ビット反転（`X`）、位相反転（`Z`）、両方（`Y`）など。`"XX"` のように 2 量子ビットの相関した誤差も書ける |
| `coherent_unitary_error(unitary)` | 量子 | ユニタリ行列 | 確率的でない系統誤差。回転角のずれ（過回転）など。例：`RXGate(0.01).to_matrix()` |
| `mixed_unitary_error([(U1, p1), (U2, p2), ...])` | 量子 | ユニタリ行列と確率の組 | 確率で選ばれるユニタリ。パウリ以外の確率的な誤差 |
| `reset_error(prob0, prob1=0)` | 量子 | 確率 | 確率で $\|0\rangle$（または $\|1\rangle$）にリセットされる。状態準備の誤りなど |
| `kraus_error(noise_ops)` | 量子 | Kraus 演算子のリスト | 任意の量子チャネル。上にない誤差を自分で定義する |
| `ReadoutError(probabilities)` | 古典（測定時） | 混同行列 | 測定値の読み間違い。非対称（$P(1\mid 0) \ne P(0\mid 1)$）も表せる |
-->

| Function (`qiskit_aer.noise`) | Type | Main arguments | What it models |
|---|---|---|---|
| `depolarizing_error(param, num_qubits)` | Quantum (after gates) | Depolarizing parameter λ, number of qubits | Replaces the state with the maximally mixed state with probability λ. The standard model summarizing gate infidelity |
| `thermal_relaxation_error(t1, t2, time, excited_state_population=0)` | Quantum | $T_1$, $T_2$, time | Energy relaxation ($\|1\rangle \to \|0\rangle$) and dephasing during a gate time or idle time |
| `amplitude_damping_error(param_amp, excited_state_population=0)` | Quantum | Damping probability γ | Energy relaxation only ($\|1\rangle \to \|0\rangle$) |
| `phase_damping_error(param_phase)` | Quantum | Dephasing strength | Dephasing only. Weakens the interference of superpositions but does not change the measurement probabilities in the computational basis |
| `phase_amplitude_damping_error(param_amp, param_phase, excited_state_population=0)` | Quantum | γ and the dephasing strength | Both of the above at once |
| `pauli_error([("X", p), ("I", 1 - p)])` | Quantum | Pairs of Pauli operator and probability | Bit flip (`X`), phase flip (`Z`), both (`Y`), and so on. Correlated 2-qubit errors such as `"XX"` can also be written |
| `coherent_unitary_error(unitary)` | Quantum | Unitary matrix | A non-random systematic error, such as an over-rotation. Example: `RXGate(0.01).to_matrix()` |
| `mixed_unitary_error([(U1, p1), (U2, p2), ...])` | Quantum | Pairs of unitary and probability | A unitary chosen at random. Random errors other than Paulis |
| `reset_error(prob0, prob1=0)` | Quantum | Probabilities | Resets to $\|0\rangle$ (or $\|1\rangle$) with some probability, e.g., state-preparation errors |
| `kraus_error(noise_ops)` | Quantum | List of Kraus operators | Any quantum channel. For defining an error not listed above |
| `ReadoutError(probabilities)` | Classical (at measurement) | Confusion matrix | Misreading of the measured value. Can be asymmetric ($P(1\mid 0) \ne P(0\mid 1)$) |

<!--
| `NoiseModel` のメソッド | 内容 |
|---|---|
| `add_all_qubit_quantum_error(error, instructions)` | すべての量子ビットの、指定したゲートの後に誤差を加える（このノートブックの `build_noise_model` の方法） |
| `add_quantum_error(error, instructions, qubits)` | 特定の量子ビット（の組）だけに誤差を加える。量子ビットごとに違う誤差や、特定の組の相関した誤差に使う |
| `add_all_qubit_readout_error(error)` / `add_readout_error(error, qubits)` | すべての / 特定の量子ビットの測定に読み出し誤差を加える |
| `NoiseModel.from_backend(backend)` | 実機（または実機を模した backend）のキャリブレーションの値からノイズモデルを作る |
-->

| `NoiseModel` method | What it does |
|---|---|
| `add_all_qubit_quantum_error(error, instructions)` | Adds an error after the specified gates on all qubits (the method used by `build_noise_model` in this notebook) |
| `add_quantum_error(error, instructions, qubits)` | Adds an error only on specific qubits (or pairs of qubits). For errors that differ by qubit, or correlated errors on a specific pair |
| `add_all_qubit_readout_error(error)` / `add_readout_error(error, qubits)` | Adds a readout error to the measurement of all / specific qubits |
| `NoiseModel.from_backend(backend)` | Builds a noise model from the calibration values of a real device (or a backend that mimics one) |

<!--
`NoiseModel` では表せないものもあります。

- **shot どうしの相関**：各 shot は独立にシミュレーションされるので、前の測定結果が次の結果に影響するような相関は作れません。測定後のビット列を加工する必要があります。
- **時間とともに変わるノイズ（ドリフト）**：1 回の実行の間、ノイズモデルは変わりません。測定を分けて、強さを変えながら実行する必要があります。
- **量子ビットの外への漏れ（リーク）**：量子ビットの 2 準位の外の状態は扱えません。
-->

Some things cannot be expressed with `NoiseModel`.

- **Correlations between shots**: each shot is simulated independently, so correlations in which one measurement result affects the next cannot be created. The measured bit string has to be processed afterwards.
- **Noise that changes over time (drift)**: the noise model does not change during one run. The measurement has to be split into parts run with different strengths.
- **Leakage out of the qubit**: states outside the two levels of the qubit cannot be represented.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `build_noise_model` | ゲートの誤差と読み出し誤差を加えた NoiseModel | `build_noise_model("both")`、`build_noise_model({"thermal": 1000})` |
| `noise_components` | 確かめた `{成分: 強さ}` の辞書。知らない成分や範囲外の強さならエラー | `noise_components("both")` |
| `readout_confusion` | 読み出しの成分の混同行列を掛け合わせた 2×2 の numpy 配列。読み出しの成分がなければ `None` | `readout_confusion({"readout": 0.03})` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `build_noise_model` | NoiseModel with gate errors and readout errors | `build_noise_model("both")`, `build_noise_model({"thermal": 1000})` |
| `noise_components` | The checked dictionary `{component: strength}`. An error is raised for an unknown component or a strength out of range | `noise_components("both")` |
| `readout_confusion` | 2×2 numpy array: the product of the confusion matrices of the readout components. `None` if there are no readout components | `readout_confusion({"readout": 0.03})` |

<br>

<!--
| 関数 | 引数 | 入力する値 | 省略時 |
|---|---|---|---|
| `build_noise_model`, `noise_components`, `readout_confusion` | `kind` | `NOISE_KINDS` の名前（`"gate"`, `"readout"`, `"both"` など）、または `{成分の名前: 強さ}` の辞書。強さは 0 以上で、成分の `max` 以下 | 必須 |
-->

| Function | Argument | Value to pass | If omitted |
|---|---|---|---|
| `build_noise_model`, `noise_components`, `readout_confusion` | `kind` | A name in `NOISE_KINDS` (`"gate"`, `"readout"`, `"both"`, and so on), or a dictionary `{component name: strength}`. Each strength must be at least 0 and at most the component's `max` | Required |

<br>

<!--
`NOISE_COMPONENTS` の各成分に書く項目は次のとおりです。

| 項目 | 内容 | 省略時 |
|---|---|---|
| `label` | 強さの意味。5.9 のグラフの横軸の名前にもなる | 必須 |
| `max` | 強さの上限。`None` なら上限なし | `None` |
| `gate_errors` | 強さ → `{(ゲートの名前, ...): QuantumError}` の関数。誤差をすべての量子ビットのそのゲートの後に加える | なし |
| `readout` | 強さ → 混同行列 `[[P(0|0), P(1|0)], [P(0|1), P(1|1)]]` の関数。すべての量子ビットの測定に加える | なし |
| `log_scale` | `True` なら、5.9 のグラフの横軸を、0〜1 は線形、1 以上は対数の目盛り（symlog）にする | `False` |
-->

The items written for each component in `NOISE_COMPONENTS` are as follows.

| Item | Contents | If omitted |
|---|---|---|
| `label` | What the strength means. Also used as the horizontal-axis label in 5.9 | Required |
| `max` | Upper limit of the strength. `None` means no upper limit | `None` |
| `gate_errors` | Function strength → `{(gate name, ...): QuantumError}`. The error is added after those gates on all qubits | None |
| `readout` | Function strength → confusion matrix `[[P(0|0), P(1|0)], [P(0|1), P(1|1)]]`. Added to the measurement of all qubits | None |
| `log_scale` | If `True`, the horizontal axis in 5.9 is linear from 0 to 1 and logarithmic above 1 (symlog) | `False` |

In [ ]:
# ---- Noise components ----
# Each component is one kind of noise whose strength can be set. To add a new kind of noise,
# add an entry here (the table in 5.2 lists the errors Qiskit Aer provides).
#   label       : what the strength means (also the horizontal-axis label in 5.9)
#   max         : largest allowed strength (None: no upper limit)
#   gate_errors : function strength -> {(gate names, ...): QuantumError}; the error is added after those gates on all qubits
#   readout     : function strength -> confusion matrix [[P(0|0), P(1|0)], [P(0|1), P(1|1)]] for all measurements
#   log_scale   : True to draw the horizontal axis in 5.9 on a symlog scale
NOISE_COMPONENTS = {
    "depolarizing": dict(
        label="1q depolarizing parameter lambda", max=1,
        gate_errors=lambda s: {("rx", "ry"): depolarizing_error(s, 1)}),
    "cz_depolarizing": dict(
        label="CZ depolarizing parameter lambda_CZ", max=1,
        gate_errors=lambda s: {("cz",): depolarizing_error(s, 2)}),
    "thermal": dict(
        label="Thermal gate-time multiplier", max=None, log_scale=True,
        gate_errors=lambda s: {
            ("rx", "ry"): thermal_relaxation_error(T1, T2, s * GATE_TIME),
            # CZ gates: thermal relaxation on both qubits, placed side by side with tensor
            ("cz",): thermal_relaxation_error(T1, T2, s * CZ_GATE_TIME).tensor(
                thermal_relaxation_error(T1, T2, s * CZ_GATE_TIME)),
        }),
    "readout": dict(
        label="Symmetric readout error epsilon", max=1,
        readout=lambda s: [[1 - s, s], [s, 1 - s]]),
    "readout_asym": dict(
        label="Asymmetric readout error P(0|1) = delta", max=1,
        readout=lambda s: [[1, 0], [s, 1 - s]]),
}

# ---- Noise kinds used in 5.3-5.8: name -> {component: strength} ----
# Components are applied in the order written (here: depolarizing, then thermal relaxation).
# The strength of "thermal" is the multiplier of GATE_TIME and CZ_GATE_TIME (1 = the gate times in 5.1).
NOISE_KINDS = {
    "gate": {"depolarizing": DEPOLARIZING_PARAM, "cz_depolarizing": CZ_DEPOLARIZING_PARAM, "thermal": 1},
    "readout": {"readout": READOUT_ERROR_RATE},
    "both": {"depolarizing": DEPOLARIZING_PARAM, "cz_depolarizing": CZ_DEPOLARIZING_PARAM, "thermal": 1,
             "readout": READOUT_ERROR_RATE},
}


def noise_components(kind):
    # A name in NOISE_KINDS, or a dictionary {component: strength} -> the checked dictionary {component: strength}
    components = NOISE_KINDS.get(kind) if isinstance(kind, str) else kind
    if components is None:
        raise ValueError(f"kind must be one of {list(NOISE_KINDS)} or a dictionary {{component: strength}} (got: {kind!r})")
    for name, strength in components.items():
        if name not in NOISE_COMPONENTS:
            raise ValueError(f"Unknown noise component: {name!r} (defined: {list(NOISE_COMPONENTS)})")
        upper = NOISE_COMPONENTS[name].get("max")
        if not math.isfinite(strength) or strength < 0 or (upper is not None and strength > upper):
            raise ValueError(f"The strength of {name!r} must be between 0 and {upper} (got: {strength})")
    return components


def readout_confusion(kind):
    # Product of the confusion matrices of the readout components (None if there are none)
    confusion = None
    for name, strength in noise_components(kind).items():
        make = NOISE_COMPONENTS[name].get("readout")
        if make is None or strength == 0:
            continue
        matrix = np.array(make(strength), dtype=float)
        confusion = matrix if confusion is None else confusion @ matrix
    return confusion


def build_noise_model(kind):
    noise_model = NoiseModel(basis_gates=BASIS_GATES)
    # Errors on the same gate are composed in the order of the components and added once
    gate_errors = {}
    for name, strength in noise_components(kind).items():
        make = NOISE_COMPONENTS[name].get("gate_errors")
        if make is None or strength == 0:
            continue
        for gates, error in make(strength).items():
            for gate in gates:
                gate_errors[gate] = error if gate not in gate_errors else gate_errors[gate].compose(error)
    for gate, error in gate_errors.items():
        noise_model.add_all_qubit_quantum_error(error, [gate])
    # Readout errors are combined into one confusion matrix (adding a second ReadoutError would replace the first)
    confusion = readout_confusion(kind)
    if confusion is not None:
        noise_model.add_all_qubit_readout_error(ReadoutError(confusion))
    return noise_model


for kind in NOISE_KINDS:
    print(f"===== {kind} =====")
    print(build_noise_model(kind))

<!-- ### 5.3 ゲート・ノイズの種類・測定回数を選ぶ -->

### 5.3 Choosing the gate, noise type, and number of measurements

<!-- `GATE_NOISY` に `"H"`, `"Rx"`, `"Ry"` のいずれかを、`NOISE_KIND` に `"gate"`, `"readout"`, `"both"` のいずれか（または 5.2 の `NOISE_KINDS` に自分で加えた名前）を指定します。測定回数は 4. と同じ $10^6$ shots です。 -->

Set `GATE_NOISY` to one of `"H"`, `"Rx"`, `"Ry"`, and set `NOISE_KIND` to one of `"gate"`, `"readout"`, `"both"` (or a name you added to `NOISE_KINDS` in 5.2). The number of measurements is $10^6$ shots, the same as in Section 4.

In [ ]:
GATE_NOISY = "H"          # Choose from "H", "Rx", "Ry"
NOISE_KIND = "both"       # A name in NOISE_KINDS (5.2): "gate", "readout", "both", ...
SHOTS_NOISY = 1_000_000   # Number of measurements

noise_model = build_noise_model(NOISE_KIND)

<!-- ### 5.4 回路を作って transpile する -->

### 5.4 Building and transpiling the circuit

<!-- 4.2 と同じです。ノイズはゲート `rx`, `ry`, `cz` に加えるので、transpile 後のゲートの数がそのまま「ノイズを受ける回数」になります（説明パート D の表）。 -->

This is the same as 4.2. Since noise is added at the `rx`, `ry`, and `cz` gates, the number of gates after transpilation is exactly the "number of times noise is received" (the table in Explanation Part D).

In [ ]:
qc_noisy = build_c3(GATE_NOISY)
qc_noisy_t = transpile(qc_noisy, basis_gates=BASIS_GATES, optimization_level=0)

print(qc_noisy_t.draw(fold=-1))
ops = qc_noisy_t.count_ops()
print("Number of 1-qubit gates receiving noise:", ops.get("rx", 0) + ops.get("ry", 0))
print("Number of CZ gates receiving noise     :", ops.get("cz", 0))

<!-- ### 5.5 測定する -->

### 5.5 Measuring

<!-- 4.3 との違いは、`AerSimulator(noise_model=noise_model)` でノイズモデルを渡すところだけです。ノイズによって $|00000\rangle$, $|11111\rangle$ 以外の結果が現れるので、多いものから 10 通りを表示します。 -->

The only difference from 4.3 is that the noise model is passed in via `AerSimulator(noise_model=noise_model)`. Since noise causes outcomes other than $|00000\rangle$ and $|11111\rangle$ to appear, the 10 most frequent outcomes are displayed.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、このセルの raw dataを `5.5_noisy_single/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data of this cell is saved in `5.5_noisy_single/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
backend_noisy = AerSimulator(noise_model=noise_model, seed_simulator=SEED)

job = backend_noisy.run(qc_noisy_t, shots=SHOTS_NOISY, memory=True)
result = job.result()

counts_noisy = result.get_counts()
raw_shots_noisy = result.get_memory()
decoded_bits_noisy = "".join(decode_c3(s) for s in raw_shots_noisy)

print("Number of distinct outcomes =", len(counts_noisy))
print()
print("Top 10 outcomes")
print("string | count   | ratio   |  # of 1s | majority")
for x, n in sorted(counts_noisy.items(), key=lambda item: item[1], reverse=True)[:10]:
    print(f"{x}  | {n:7d} | {n / SHOTS_NOISY:.5f} | {hamming_weight(x):8d} | {decode_c3(x)}")

print()
print("Bit string from the majority vote (first 50 bits) =", decoded_bits_noisy[:50])

# Save the raw data (only when SAVE_RESULTS = True)
if SAVE_RESULTS:
    path = save_raw(output_dir("5.5_noisy_single") / f"c3_{GATE_NOISY}_noise-{NOISE_KIND}.txt", raw_shots_noisy,
                    GATE_NOISY, SHOTS_NOISY, SEED, noise_model, metadata={"condition": f"noise-{NOISE_KIND}"})
    print("Saved:", path)

<!-- ### 5.6 評価指標を計算する -->

### 5.6 Computing the evaluation metrics

<!-- 4.4 と同じ指標を計算し、説明パート C・E の予想と比べます。 -->

We compute the same metrics as in 4.4 and compare them with the predictions of Explanation Parts C and E.

<!--
- **non-GHZ の割合**：`NOISE_KIND` が `"readout"` なら理論値 $1 - (1-\varepsilon)^5 - \varepsilon^5 \approx 0.141$ と一致するはずです。`"gate"` では約 0.051、`"both"` では約 0.184 です（5.9 の密度行列による計算値）。
- **多数決で得たビット**：誤りの大部分が訂正され、bias は揺らぎの目安と同程度のはずです。
- **5 ビットの結果の min-entropy**：ノイズで 1 より大きくなりますが、増えた分は誤りによるものです（説明パート C の (3)）。
-->

- **Non-GHZ fraction**: If `NOISE_KIND` is `"readout"`, it should match the theoretical value $1 - (1-\varepsilon)^5 - \varepsilon^5 \approx 0.141$. For `"gate"` it is about 0.051, and for `"both"` about 0.184 (values calculated with the density matrix in 5.9).
- **Bit from the majority vote**: Most of the errors should be corrected, and the bias should be comparable to the reference value for the fluctuation.
- **Min-entropy of the 5-bit outcomes**: Noise makes it larger than 1, but the increase is due to errors (item (3) of Explanation Part C).

In [ ]:
r = evaluate_c3(counts_noisy)

print("===== Distribution of the number of 1s =====")
show_weights(r)
print(f"non-GHZ fraction = {r['non_ghz']:.5f}")
if NOISE_KIND in ("readout", "both"):
    eps = READOUT_ERROR_RATE
    print(f"[Theory] non-GHZ fraction from readout error alone = {readout_non_ghz(eps):.5f}")
    print(f"[Theory] majority-vote error probability M(ε) with readout error only = {majority_error_prob(eps):.2e}"
          f" (for 1 qubit, ε = {eps})")

print()
print("===== Bits from the majority vote =====")
show_metrics(r)

print()
print("===== 5-bit outcomes =====")
print(f"min-entropy (5-bit distribution) = {r['raw_min_entropy']:.6f} (ideal: 1)")

print()
print("===== (Optional) Postselection =====")
print(f"P(1) using only 00000 and 11111 = {r['postselected_P1']:.6f}"
      f" (fraction of usable shots {1 - r['non_ghz']:.5f})")

<!-- ### 5.7 測定を繰り返す -->

### 5.7 Repeating the measurement

<!-- 4.5 の `run_trials` にノイズモデルを渡して、`N_TRIALS` 回繰り返します。 -->

We pass the noise model to `run_trials` from 4.5 and repeat `N_TRIALS` times.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、試行ごとのファイル（`c3_H_noise-both_trial001.txt` など）と `trials.csv`を `5.7_noisy_trials/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., one file per trial (`c3_H_noise-both_trial001.txt` and so on) and `trials.csv` are saved in `5.7_noisy_trials/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
rows_noisy = run_trials(GATE_NOISY, SHOTS_NOISY, N_TRIALS, noise_model=noise_model, seed=SEED,
                        raw_dir=output_dir("5.7_noisy_trials"), label=f"noise-{NOISE_KIND}")
show_table(rows_noisy, "trial")
summarize(rows_noisy)

<!-- ### 5.8 （参考）ゲートとノイズの種類をすべて比べる -->

### 5.8 (Optional) Comparing all gates and noise types

<!-- 3 つのゲート × 3 種類のノイズの 9 通りを 1 回ずつ測定して並べます（$10^6$ shots × 9 回）。量子ビット 2 のゲートが違っても、CX ゲートの部分は共通なので、non-GHZ の割合はノイズの種類でほぼ決まります。 -->

We measure all 9 combinations of 3 gates × 3 noise types once each and list the results side by side ($10^6$ shots × 9 runs). Even if the gate on qubit 2 differs, the CX part is common, so the non-GHZ fraction is almost entirely determined by the noise type.

<!-- 1. で `SAVE_RESULTS = True` にした場合は、9 通りの raw data（`c3_H_noise-gate.txt` など）と一覧 `summary.csv`を `5.8_noisy_comparison/` に保存します（「G. このノートブックの使い方」の「保存されるファイル」を参照）。 -->

If `SAVE_RESULTS = True` in 1., the raw data for the 9 combinations (`c3_H_noise-gate.txt` and so on) and their list, `summary.csv`, are saved in `5.8_noisy_comparison/` (see "Saved files" in "G. How to use this notebook").

In [ ]:
conditions = [(gate, f"noise-{kind}", build_noise_model(kind)) for kind in NOISE_KINDS for gate in GATES]
rows = run_conditions(conditions, SHOTS_NOISY, "5.8_noisy_comparison")
show_table(rows, "gate / noise")
print(f"[Reference] bias = {expected_bias(SHOTS_NOISY):.6f}")

<!-- ### 5.9 （参考）ノイズの大きさを段階的に変える -->

### 5.9 (Optional) Changing the noise strength step by step

<!-- C1 のノートブックの 5.9 と同じく、ノイズを**1 成分ずつ**変更し、ほかの成分をゼロにして原因を切り分けます。各条件で $H_{\mathrm{Gate}}$ / Rx / Ry を比較します。C3 回路では、C1 の 3 成分に CZ ゲートの脱分極と非対称な読み出し誤差を加えた 5 成分を調べます。 -->

As in 5.9 of the C1 notebook, we vary the noise **one component at a time**, setting the other components to zero to isolate the cause. Under each condition we compare $H_{\mathrm{Gate}}$ / Rx / Ry. For the C3 circuit, we examine 5 components: the 3 components of C1 plus CZ gate depolarizing and asymmetric readout error.

<!--
- 1 量子ビットゲートの脱分極：`rx`, `ry` の脱分極パラメータ λ を 0〜0.10 で変えます。
- CZ ゲートの脱分極：`cz` の 2 量子ビット脱分極パラメータ $\lambda_{\mathrm{CZ}}$ を 0〜0.20 で変えます。
- 熱緩和：T1、T2 は 5.1 の値に固定し、ノイズに渡す有効ゲート時間を `倍率 × GATE_TIME`（1 量子ビットゲート）、`倍率 × CZ_GATE_TIME`（CZ ゲート）にします。倍率 0 は熱緩和なし、1 は元の条件です。大きい倍率は影響を観察するための仮想条件です。
- 対称な読み出し誤差：0→1 と 1→0 を読み間違える確率 $\varepsilon$ を 0〜0.20 で変えます。
- 非対称な読み出し誤差：$P(1|0) = 0$ に固定し、$P(0|1) = \delta$ を 0〜0.20 で変えます。説明パート C の「多数決が偏りを抑えるしくみ」を確かめるためのものです。
-->

- 1-qubit gate depolarizing: vary the depolarizing parameter λ of `rx` and `ry` from 0 to 0.10.
- CZ gate depolarizing: vary the 2-qubit depolarizing parameter $\lambda_{\mathrm{CZ}}$ of `cz` from 0 to 0.20.
- Thermal relaxation: fix T1 and T2 at the values in 5.1, and set the effective gate time passed to the noise to `multiplier × GATE_TIME` (1-qubit gates) and `multiplier × CZ_GATE_TIME` (CZ gates). A multiplier of 0 means no thermal relaxation, and 1 is the original condition. Large multipliers are hypothetical conditions for observing the effect.
- Symmetric readout error: vary the probability $\varepsilon$ of misreading 0→1 and 1→0 from 0 to 0.20.
- Asymmetric readout error: fix $P(1|0) = 0$ and vary $P(0|1) = \delta$ from 0 to 0.20. This is to verify "How the majority vote suppresses bias" in Explanation Part C.

<!-- 縦軸は、多数決で得たビットの $P(1)$・bias・min-entropy と、non-GHZ の割合の 4 つです。 -->

The vertical axes are four quantities: $P(1)$, bias, and min-entropy of the bit from the majority vote, and the non-GHZ fraction.

<!-- `SWEEP_VALUES` のリストを編集すると、範囲と刻み幅を自由に変更できます。$10^6$ shots × 3 試行で確認します。全条件数と総 shots は実行前に表示します。条件数が多いので、実行には数分〜十数分かかります。短時間で試すときは `SWEEP_SHOTS` を小さくしてください。 -->

The range and step size can be changed freely by editing the lists in `SWEEP_VALUES`. We check with $10^6$ shots × 3 trials. The total number of conditions and total shots are displayed before running. Because there are many conditions, the run takes from several minutes to over ten minutes. Reduce `SWEEP_SHOTS` for a quick test.

<!-- ノイズの成分は 5.2 の `NOISE_COMPONENTS` で定義しています。新しい成分を 5.2 で加えたあと、`SWEEP_VALUES` に成分の名前と強さのリストを加えると、その成分も強さを変えて測定します。グラフの列は成分の数だけ自動で増えます。 -->

The noise components are defined in `NOISE_COMPONENTS` in 5.2. After adding a new component in 5.2, add its name and a list of strengths to `SWEEP_VALUES` to vary its strength as well. The plot gets one column per component automatically.

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `build_sweep_noise` | 指定した 1 成分だけを含む NoiseModel | `build_sweep_noise("readout_asym", 0.1)` |
| `sweep_exact` | 測定揺らぎを含まない、モデルから計算した指標の辞書（`P1`・`bias`・`min_entropy`・`non_ghz`） | `sweep_exact("H", "readout_asym", 0.1, build_sweep_noise("readout_asym", 0.1))` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `build_sweep_noise` | NoiseModel containing only the specified single component | `build_sweep_noise("readout_asym", 0.1)` |
| `sweep_exact` | Dictionary of metrics computed from the model, without measurement fluctuation (`P1`, `bias`, `min_entropy`, `non_ghz`) | `sweep_exact("H", "readout_asym", 0.1, build_sweep_noise("readout_asym", 0.1))` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>build_sweep_noise</code></td><td><code>component</code></td><td>5.2 の <code>NOISE_COMPONENTS</code> にある成分の名前で、<code>SWEEP_VALUES</code> のキーにもあるもの（例：<code>"thermal"</code>）</td><td>必須</td></tr>
<tr><td><code>strength</code></td><td>有限の非負値。脱分極は λ（0〜1）、読み出しは誤り確率（0〜1）、熱緩和はゲート時間の倍率（例：<code>1000</code>）。<code>0</code> はノイズなし</td><td>必須</td></tr>
<tr><td rowspan="4"><code>sweep_exact</code></td><td><code>gate</code></td><td><code>"H"</code>・<code>"Rx"</code>・<code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>component</code></td><td><code>build_sweep_noise</code> と同じノイズ成分名</td><td>必須</td></tr>
<tr><td><code>strength</code></td><td><code>build_sweep_noise</code> に渡したものと同じ強さ。読み出し誤差を確率に反映するときに使用</td><td>必須</td></tr>
<tr><td><code>nm</code></td><td>同じ <code>component</code>・<code>strength</code> で <code>build_sweep_noise</code> が返した NoiseModel</td><td>必須</td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td rowspan="2"><code>build_sweep_noise</code></td><td><code>component</code></td><td>A component name in <code>NOISE_COMPONENTS</code> (5.2) that is also a key of <code>SWEEP_VALUES</code> (e.g., <code>"thermal"</code>)</td><td>Required</td></tr>
<tr><td><code>strength</code></td><td>A finite non-negative value. λ (0–1) for depolarizing, error probability (0–1) for readout, gate-time multiplier (e.g., <code>1000</code>) for thermal relaxation. <code>0</code> means no noise</td><td>Required</td></tr>
<tr><td rowspan="4"><code>sweep_exact</code></td><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>component</code></td><td>Same noise component name as for <code>build_sweep_noise</code></td><td>Required</td></tr>
<tr><td><code>strength</code></td><td>Same strength as passed to <code>build_sweep_noise</code>. Used to reflect the readout error in the probabilities</td><td>Required</td></tr>
<tr><td><code>nm</code></td><td>NoiseModel returned by <code>build_sweep_noise</code> for the same <code>component</code> and <code>strength</code></td><td>Required</td></tr>
</tbody>
</table>

<br>

<!-- `sweep_exact` は、測定直前の密度行列の対角成分から 32 通りの確率を求め、読み出し誤差があるときは各量子ビットに混同行列をかけて、測定される確率を計算します。その確率を `evaluate_c3` に渡して指標を求めます。 -->

`sweep_exact` obtains the probabilities of the 32 outcomes from the diagonal elements of the density matrix just before measurement and, if there is readout error, applies the confusion matrix to each qubit to compute the probabilities that are measured. Those probabilities are passed to `evaluate_c3` to obtain the metrics.


In [ ]:
SWEEP_VALUES = {
    "depolarizing": [i / 100 for i in range(11)],
    "cz_depolarizing": [i / 50 for i in range(11)],
    "thermal": [0, 1, 10, 100, 250, 500, 1000, 2500, 5000],
    "readout": [i / 50 for i in range(11)],
    "readout_asym": [i / 50 for i in range(11)],
}
SWEEP_GATES = list(GATES)
SWEEP_SHOTS = 1_000_000
SWEEP_TRIALS = 3
SWEEP_SEED = 2026


def build_sweep_noise(component, strength):
    # NoiseModel with only one component (all other components are zero)
    if component not in SWEEP_VALUES:
        raise ValueError(f"Unknown noise component: {component} (add it to SWEEP_VALUES)")
    return build_noise_model({component: strength})


def apply_readout(probs, confusion):
    # probs: probabilities of length 32 (the binary expansion of index k is q4 ... q0). Apply the confusion matrix to each qubit
    p = np.asarray(probs, dtype=float).reshape((2,) * N_QUBITS)
    for axis in range(N_QUBITS):
        p = np.moveaxis(np.tensordot(p, confusion, axes=([axis], [0])), -1, axis)
    return p.reshape(-1)


def sweep_exact(gate, component, strength, nm):
    # Save the density matrix before measurement. Readout error is applied with a classical confusion matrix.
    qc = transpile(build_c3(gate), basis_gates=BASIS_GATES, optimization_level=0)
    qc = qc.remove_final_measurements(inplace=False)
    qc.save_density_matrix()
    backend = AerSimulator(method="density_matrix", noise_model=nm)
    rho = backend.run(qc, shots=1).result().data(0)["density_matrix"]
    probs = np.clip(np.real(np.diag(rho.data)), 0, 1)
    confusion = readout_confusion({component: strength})
    if confusion is not None:
        probs = apply_readout(probs, confusion)
    probs = probs / probs.sum()
    counts = {format(k, f"0{N_QUBITS}b"): float(p) for k, p in enumerate(probs)}  # Pass probabilities as 'counts'
    r = evaluate_c3(counts)
    return {key: r[key] for key in ("P1", "bias", "min_entropy", "non_ghz")}

<!-- **次のコードセルで定義する関数の使い方** -->

**How to use the functions defined in the next code cell**

<!-- 既定値がある引数は省略できます。使用例は、次のコードセルを実行した後に呼び出してください。 -->

Arguments with default values can be omitted. Call the examples after running the next code cell.

<!--
| 関数 | 戻り値・処理 | 使用例 |
|---|---|---|
| `run_noise_sweep` | `(試行ごとの記録リスト, 条件ごとの要約リスト)` のタプル。`raw_dir` を指定すると、raw data と `trials.csv`・`summary.csv` をそのフォルダに保存する | `records, summary = run_noise_sweep(raw_dir="results/c3/2026-11-10_153012/5.9_noise_sweep")` |
| `measure_condition` | `(trials.csv の行のリスト, summary.csv の 1 行)`。1 つの条件で `SWEEP_TRIALS` 回測定し、密度行列からの値と平均・標準偏差をまとめる（`run_noise_sweep` の中で使う） | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |
-->

| Function | Return value / behavior | Example |
|---|---|---|
| `run_noise_sweep` | Tuple `(list of per-trial records, list of per-condition summaries)`. If `raw_dir` is given, the raw data and `trials.csv` and `summary.csv` are saved in that folder | `records, summary = run_noise_sweep(raw_dir="results/c3/2026-11-10_153012/5.9_noise_sweep")` |
| `measure_condition` | `(rows for trials.csv, one row for summary.csv)`. Measures `SWEEP_TRIALS` times under one condition and collects the density-matrix values, means, and standard deviations (used inside `run_noise_sweep`) | `measure_condition("thermal", 1000, "H", build_sweep_noise("thermal", 1000), None, 2026)` |

<br>

<!--
<table>
<thead>
<tr><th>関数</th><th>引数</th><th>入力する値</th><th>省略時</th></tr>
</thead>
<tbody>
<tr><td><code>run_noise_sweep</code></td><td><code>raw_dir</code></td><td>条件・試行別の raw data と、結果の CSV（<code>trials.csv</code>, <code>summary.csv</code>）の保存フォルダ。文字列または Path。<code>None</code> なら raw data も CSV も保存しない。空でないフォルダを指定すると測定前にエラーになるので、再実行時は未使用の保存先を指定</td><td><code>None</code></td></tr>
<tr><td rowspan="6"><code>measure_condition</code></td><td><code>component</code>, <code>strength</code></td><td>ノイズの成分の名前と強さ</td><td>必須</td></tr>
<tr><td><code>gate</code></td><td><code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code> のいずれか</td><td>必須</td></tr>
<tr><td><code>nm</code></td><td><code>build_sweep_noise(component, strength)</code> の NoiseModel</td><td>必須</td></tr>
<tr><td><code>raw_dir</code></td><td>raw data の保存先（<code>None</code> なら保存しない）</td><td>必須</td></tr>
<tr><td><code>first_seed</code></td><td>1 回目の試行のシード。試行ごとに 1 ずつ増やす（<code>None</code> なら毎回違う結果）</td><td>必須</td></tr>
<tr><td colspan="2">試行の回数・測定回数は <code>SWEEP_TRIALS</code>・<code>SWEEP_SHOTS</code> で決まる</td><td></td></tr>
</tbody>
</table>
-->

<table>
<thead>
<tr><th>Function</th><th>Argument</th><th>Value to pass</th><th>If omitted</th></tr>
</thead>
<tbody>
<tr><td><code>run_noise_sweep</code></td><td><code>raw_dir</code></td><td>Folder in which to save the raw data for each condition and trial, and the CSV files of the results (<code>trials.csv</code>, <code>summary.csv</code>), as a string or Path. <code>None</code> means neither the raw data nor the CSV files are saved. A non-empty folder raises an error before measuring, so specify an unused destination when re-running</td><td><code>None</code></td></tr>
<tr><td rowspan="6"><code>measure_condition</code></td><td><code>component</code>, <code>strength</code></td><td>Name and strength of the noise component</td><td>Required</td></tr>
<tr><td><code>gate</code></td><td>One of <code>"H"</code>, <code>"Rx"</code>, <code>"Ry"</code></td><td>Required</td></tr>
<tr><td><code>nm</code></td><td>NoiseModel from <code>build_sweep_noise(component, strength)</code></td><td>Required</td></tr>
<tr><td><code>raw_dir</code></td><td>Destination of the raw data (<code>None</code>: not saved)</td><td>Required</td></tr>
<tr><td><code>first_seed</code></td><td>Seed of the first trial, increased by 1 for each trial (<code>None</code>: different results each time)</td><td>Required</td></tr>
<tr><td colspan="2">The number of trials and measurements are set by <code>SWEEP_TRIALS</code> and <code>SWEEP_SHOTS</code></td><td></td></tr>
</tbody>
</table>

<br>

<!-- 引数以外の実験条件は直前の設定セルで変更します。 -->

Experimental conditions other than the arguments are changed in the preceding settings cell.

<!--
| 設定変数 | 入力する値 |
|---|---|
| `SWEEP_VALUES` | 成分名（5.2 の `NOISE_COMPONENTS` の名前）をキー、試す強さのリストを値とする辞書 |
| `SWEEP_GATES` | 比較するゲートのリスト（例：`["H", "Rx", "Ry"]`） |
| `SWEEP_SHOTS` | 1 試行あたりの測定回数（shot 数）。1 以上の整数 |
| `SWEEP_TRIALS` | 条件ごとの試行数。標準偏差の計算のため 2 以上の整数 |
| `SWEEP_SEED` | 非負整数または `None` |
| `SWEEP_RAW_DIR` | 次のコードセル末尾の呼び出しに渡す保存先。1. で `SAVE_SWEEP = True` なら `RUN_DIR/5.9_noise_sweep`、`False` なら `None` に自動で決まる |
-->

| Setting variable | Value to set |
|---|---|
| `SWEEP_VALUES` | Dictionary with component names (names in `NOISE_COMPONENTS` in 5.2) as keys and lists of strengths to try as values |
| `SWEEP_GATES` | List of gates to compare (e.g., `["H", "Rx", "Ry"]`) |
| `SWEEP_SHOTS` | Number of measurements (shots) per trial. An integer of at least 1 |
| `SWEEP_TRIALS` | Number of trials per condition. An integer of at least 2, for computing the standard deviation |
| `SWEEP_SEED` | Non-negative integer or `None` |
| `SWEEP_RAW_DIR` | Destination passed to the call at the end of the next code cell. Set automatically: `RUN_DIR/5.9_noise_sweep` if `SAVE_SWEEP = True` in 1., otherwise `None` |


In [ ]:
SWEEP_METRICS = ("P1", "bias", "shannon", "min_entropy", "non_ghz", "raw_min_entropy")


def measure_condition(component, strength, gate, nm, raw_dir, first_seed):
    # SWEEP_TRIALS measurements under one condition -> (rows for trials.csv, one row for summary.csv)
    rows = []
    for trial in range(1, SWEEP_TRIALS + 1):
        name = f"c3_{gate}_{component}-{strength:g}_trial{trial:03d}.txt"
        raw_path = None if raw_dir is None else Path(raw_dir) / component / name
        seed = offset_seed(first_seed, trial - 1)
        r = run_c3(gate, SWEEP_SHOTS, nm, seed, raw_path, method="density_matrix", optimization_level=0,
                   metadata={"component": component, "strength": strength, "trial": trial})
        rows.append(dict(component=component, strength=strength, gate=gate, trial=trial, seed=seed,
                         shots=r["shots"], **{k: r[k] for k in SWEEP_METRICS}))
    exact = sweep_exact(gate, component, strength, nm)
    summary = dict(component=component, strength=strength, gate=gate, **{"exact_" + k: v for k, v in exact.items()})
    for metric in SWEEP_METRICS:
        values = [row[metric] for row in rows]
        summary[metric + "_mean"] = statistics.mean(values)
        summary[metric + "_std"] = statistics.stdev(values)
    return rows, summary


def run_noise_sweep(raw_dir=None):
    if SWEEP_SHOTS < 1 or SWEEP_TRIALS < 2:
        raise ValueError("shots must be at least 1, and the number of trials must be at least 2 to compute the standard deviation")
    if raw_dir is not None and Path(raw_dir).exists() and any(Path(raw_dir).iterdir()):
        raise FileExistsError(f"Destination is not empty: {raw_dir}. Run the Setup cell (1.) again to save in a new folder.")
    conditions = len(SWEEP_GATES) * sum(map(len, SWEEP_VALUES.values()))
    print(f"{conditions} conditions × {SWEEP_TRIALS} trials × {SWEEP_SHOTS:,} shots"
          f" = {conditions * SWEEP_TRIALS * SWEEP_SHOTS:,} shots")
    records, summary = [], []
    for component, strengths in SWEEP_VALUES.items():
        for strength in strengths:
            nm = build_sweep_noise(component, strength)
            for gate in SWEEP_GATES:
                first_seed = offset_seed(SWEEP_SEED, len(records))
                rows, item = measure_condition(component, strength, gate, nm, raw_dir, first_seed)
                records += rows
                summary.append(item)
        print(f"Done: {component}")
    if raw_dir is not None:
        # The CSV files are saved only together with the raw data, after all conditions have been measured
        for name, data in [("trials.csv", records), ("summary.csv", summary)]:
            print("Saved:", write_csv(Path(raw_dir) / name, data))
    return records, summary


# Saved in RUN_DIR/5.9_noise_sweep only when SAVE_SWEEP = True in 1. Setup (None: not saved)
SWEEP_RAW_DIR = RUN_DIR / "5.9_noise_sweep" if SAVE_SWEEP else None
sweep_records, sweep_summary = run_noise_sweep(raw_dir=SWEEP_RAW_DIR)

In [ ]:
# Points and error bars: trial mean ± between-trial standard deviation (clipped to the range the metric can take)
# Solid lines: values computed from the density matrix
PLOT_ROWS = [("P1", "P(1) (decoded)", (0, 1)),
             ("bias", "Bias (decoded)", (0, 0.5)),
             ("min_entropy", "Min-entropy (decoded)", (0, 1)),
             ("non_ghz", "Non-GHZ fraction", (0, 1))]


def plot_gate(ax_col, component, gate, color):
    # One gate in one column: sampled points with error bars and the exact line, in each row
    points = sorted((r for r in sweep_summary if r["component"] == component and r["gate"] == gate),
                    key=lambda r: r["strength"])
    xs = [r["strength"] for r in points]
    for ax, (metric, ylabel, (low, high)) in zip(ax_col, PLOT_ROWS):
        means = [r[metric + "_mean"] for r in points]
        stds = [r[metric + "_std"] for r in points]
        yerr = [[min(s, m - low) for m, s in zip(means, stds)],
                [min(s, high - m) for m, s in zip(means, stds)]]
        ax.errorbar(xs, means, yerr=yerr, fmt="o", capsize=3, color=color, label=gate + " sampled")
        ax.plot(xs, [r["exact_" + metric] for r in points], color=color, label=gate + " exact")
        ax.set(xlabel=NOISE_COMPONENTS[component]["label"], ylabel=ylabel)
        ax.grid(alpha=0.3)


n_cols = len(SWEEP_VALUES)
fig, axes = plt.subplots(len(PLOT_ROWS), n_cols, figsize=(4.6 * n_cols, 14), squeeze=False)
for col, component in enumerate(SWEEP_VALUES):
    ax_col = axes[:, col]
    for gate_index, gate in enumerate(SWEEP_GATES):
        plot_gate(ax_col, component, gate, f"C{gate_index}")
    ax_col[0].axhline(0.5, color="gray", linestyle=":")
    fair_line = ax_col[1].axhline(expected_bias(SWEEP_SHOTS), color="gray", linestyle=":",
                                  label="Fair-bit expected sampled bias")
    if component == "readout_asym":
        # Theoretical values when the same readout error is applied to the C1 circuit (1 qubit, no majority vote)
        deltas = np.linspace(0, max(SWEEP_VALUES[component]), 100)
        for ax, values in zip(ax_col, [(1 - deltas) / 2, deltas / 2, -np.log2((1 + deltas) / 2)]):
            ax.plot(deltas, values, color="black", linestyle="--", label="C1 theory (single qubit)")
        ax_col[1].legend(fontsize=8)
    else:
        ax_col[1].legend(handles=[fair_line], fontsize=8)
    ax_col[0].legend(fontsize=8)
    if NOISE_COMPONENTS[component].get("log_scale"):
        # Multiplier 0 is included, so use a linear scale for 0–1 and a logarithmic scale above 1
        for ax in ax_col:
            ax.set_xscale("symlog", linthresh=1)
            ax.set_xlim(-0.3, max(SWEEP_VALUES[component]) * 1.5)
fig.tight_layout()
plt.show()

lines = ["| noise | strength | gate | P(1) mean ± SD | exact P(1) | bias mean | non-GHZ mean | exact non-GHZ |",
         "|---|---|---|---|---|---|---|---|"]
for r in sweep_summary:
    lines.append(f"| {r['component']} | {r['strength']:g} | {r['gate']} | "
                 f"{r['P1_mean']:.6f} ± {r['P1_std']:.6f} | {r['exact_P1']:.6f} | "
                 f"{r['bias_mean']:.6f} | {r['non_ghz_mean']:.5f} | {r['exact_non_ghz']:.5f} |")
display(Markdown("\n".join(lines)))

<!-- #### 結果の読み方と保存 -->

#### How to read the results and save them

<!-- **グラフは 4 行 × 5 列です。** -->

**The figure has 4 rows × 5 columns.**

<!--
| 段（縦軸） | 値の意味 |
|---|---|
| 1 段目：P(1) (decoded) | 多数決で得たビットで 1 が出る確率。偏りがないときは 0.5 |
| 2 段目：Bias (decoded) | 多数決で得たビットの P(1) と 0.5 の差の絶対値。0 に近いほど偏りが小さい |
| 3 段目：Min-entropy (decoded) | 多数決で得たビットの 1 ビットあたりの min-entropy。1 に近いほど偏りが小さい |
| 4 段目：Non-GHZ fraction | $\lvert 00000\rangle$, $\lvert 11111\rangle$ 以外の結果の割合。多数決で訂正した shot の割合 |
-->

| Row (vertical axis) | Meaning of the value |
|---|---|
| Row 1: P(1) (decoded) | Probability of getting 1 for the bit from the majority vote. 0.5 when there is no bias |
| Row 2: Bias (decoded) | Absolute difference between P(1) of the bit from the majority vote and 0.5. The closer to 0, the smaller the bias |
| Row 3: Min-entropy (decoded) | Min-entropy per bit of the bit from the majority vote. The closer to 1, the smaller the bias |
| Row 4: Non-GHZ fraction | Fraction of outcomes other than $\lvert 00000\rangle$ and $\lvert 11111\rangle$. The fraction of shots corrected by the majority vote |

<!--
| 列 | 横軸の英語表記 | 横軸の意味 |
|---|---|---|
| 1 列目：1 量子ビットゲートの脱分極 | 1q depolarizing parameter lambda | `rx`, `ry` の脱分極パラメータ λ |
| 2 列目：CZ ゲートの脱分極 | CZ depolarizing parameter lambda_CZ | `cz` の 2 量子ビット脱分極パラメータ $\lambda_{\mathrm{CZ}}$ |
| 3 列目：熱緩和 | Thermal gate-time multiplier | 熱緩和の計算に使うゲート時間の倍率。0 は熱緩和なし、1 は元の 20 ns / 60 ns。0〜1 は線形、1 以上は対数の目盛（symlog） |
| 4 列目：対称な読み出し誤差 | Symmetric readout error epsilon | 0→1 と 1→0 を読み間違える確率 $\varepsilon$ |
| 5 列目：非対称な読み出し誤差 | Asymmetric readout error P(0\|1) = delta | 1 を 0 と読み間違える確率 $\delta$（0 を 1 と読み間違えることはない） |
-->

| Column | Horizontal-axis label | Meaning of the horizontal axis |
|---|---|---|
| Column 1: 1-qubit gate depolarizing | 1q depolarizing parameter lambda | Depolarizing parameter λ of `rx` and `ry` |
| Column 2: CZ gate depolarizing | CZ depolarizing parameter lambda_CZ | 2-qubit depolarizing parameter $\lambda_{\mathrm{CZ}}$ of `cz` |
| Column 3: thermal relaxation | Thermal gate-time multiplier | Multiplier on the gate time used in the thermal relaxation calculation. 0 means no thermal relaxation, and 1 is the original 20 ns / 60 ns. Linear from 0 to 1 and logarithmic above 1 (symlog) |
| Column 4: symmetric readout error | Symmetric readout error epsilon | Probability $\varepsilon$ of misreading 0→1 and 1→0 |
| Column 5: asymmetric readout error | Asymmetric readout error P(0\|1) = delta | Probability $\delta$ of misreading 1 as 0 (0 is never misread as 1) |

<!-- **点・実線・エラーバーの見方** -->

**How to read the points, solid lines, and error bars**

<!--
- 色は H・Rx・Ry の違いを表します（凡例や表の H は $H_{\mathrm{Gate}}$ です）。
- 点（`sampled`）は有限 shots の測定を繰り返して得た指標の平均、上下の棒は試行間の標準偏差です（信頼区間ではありません）。指標が取り得る範囲で切って表示しているので、切った側の棒は標準偏差より短くなります。
- 実線（`exact`）は、設定したノイズモデルから密度行列を使って求めた値です。有限 shots による統計的な揺らぎを含みません。
- 2 段目の灰色の点線は、公平なビットを `SWEEP_SHOTS` 個測定した場合の bias の目安です。
- 5 列目の黒い破線（`C1 theory`）は、同じ非対称な読み出し誤差を C1 回路（1 量子ビット、多数決なし）に加えたときの理論値 $P(1) = (1-\delta)/2$、bias $= \delta/2$ です。
-->

- Colors distinguish H, Rx, and Ry (H in the legend and tables is $H_{\mathrm{Gate}}$).
- The points (`sampled`) are the average of the metric obtained by repeating measurements with a finite number of shots, and the bars above and below are the standard deviation between trials (they are not confidence intervals). Because they are clipped to the range the metric can take, a clipped bar is shorter than the standard deviation on that side.
- The solid lines (`exact`) are values obtained from the configured noise model using density matrices. They contain no statistical fluctuation from finite shots.
- The gray dotted line in row 2 is the reference bias when `SWEEP_SHOTS` fair bits are measured.
- The black dashed line in column 5 (`C1 theory`) is the theoretical value $P(1) = (1-\delta)/2$, bias $= \delta/2$ when the same asymmetric readout error is applied to the C1 circuit (1 qubit, no majority vote).

<!-- **結果を読む順序** -->

**Order in which to read the results**

<!--
1. まず 4 段目を見ます。どの成分でも、ノイズを強くすると non-GHZ の割合が増えます。4 列目の実線は理論値 $1 - (1-\varepsilon)^5 - \varepsilon^5$ に一致します。
2. 1・2・4 列目（脱分極と対称な読み出し誤差）では、non-GHZ の割合が大きくなっても、多数決で得たビットの $P(1)$ はほぼ 1/2 のままです。対称な誤りは公平なビットを偏らせないからです（説明パート C）。水平な線は、ノイズが偏りに現れていないことを示し、ノイズがないことを意味しません。
3. 5 列目では、C1 回路の bias（黒い破線）が $\delta/2$ で直線的に増えるのに対し、C3 回路の bias は $M(\delta)/2 \approx 5\delta^3$ なので、$\delta$ が小さい範囲ではほぼ 0 に抑えられます。これが多数決による読み出し誤差の訂正の効果です。
4. 3 列目（熱緩和）では、各量子ビットが $|0\rangle$ 側に緩和するので、倍率を大きくすると $P(1)$ が下がります。下がり方は C1 のノートブックの 5.9 より大きくなります（倍率 5000 で、C1 は $H_{\mathrm{Gate}}$ で約 0.50、Rx・Ry で約 0.45、C3 は 3 つとも約 0.25〜0.27）。量子ビット 2 は 4 個の CZ ゲートの間ずっと緩和を受け続け、そこで起きた 1→0 の緩和は、その後の CX ゲートで他の量子ビットにもコピーされるからです。このような誤りは 5 個の量子ビットで独立ではないので、多数決では訂正できません（説明パート B）。
-->

1. First look at row 4. For every component, the non-GHZ fraction increases as the noise gets stronger. The solid line in column 4 matches the theoretical value $1 - (1-\varepsilon)^5 - \varepsilon^5$.
2. In columns 1, 2, and 4 (depolarizing and symmetric readout error), even when the non-GHZ fraction grows, $P(1)$ of the bit from the majority vote remains nearly 1/2. This is because symmetric errors do not bias a fair bit (Explanation Part C). A horizontal line indicates that the noise does not show up as a bias; it does not mean that there is no noise.
3. In column 5, whereas the bias of the C1 circuit (black dashed line) grows linearly as $\delta/2$, the bias of the C3 circuit is $M(\delta)/2 \approx 5\delta^3$, so it is suppressed to nearly 0 for small $\delta$. This is the effect of readout error correction by majority vote.
4. In column 3 (thermal relaxation), each qubit relaxes toward $|0\rangle$, so $P(1)$ decreases as the multiplier increases. The decrease is larger than in 5.9 of the C1 notebook (at a multiplier of 5000, C1 gives about 0.50 for $H_{\mathrm{Gate}}$ and about 0.45 for Rx and Ry, while C3 gives about 0.25–0.27 for all three). This is because qubit 2 keeps relaxing throughout the 4 CZ gates, and a 1→0 relaxation occurring there is copied to the other qubits by the subsequent CX gates. Such errors are not independent across the 5 qubits, so they cannot be corrected by majority vote (Explanation Part B).

<!-- **このモデルの限界**：このモデルでは量子ビットごとの誤差がすべて同じで、量子ビット間の干渉もありません。5 列目のとおり、$P(0|1) > P(1|0)$ の読み出し誤差は 1 の割合を下げる方向に働くので、論文の Odra 5 で見られた「5 量子ビットの C3 回路で 1 の割合が 50% を上回る」結果は、このモデルでは再現できません（説明パート E）。 -->

**Limitations of this model**: In this model, all per-qubit errors are identical and there is no interference between qubits. As column 5 shows, readout error with $P(0|1) > P(1|0)$ acts to lower the fraction of 1s, so the result seen on Odra 5 in the paper, "the fraction of 1s exceeds 50% for the 5-qubit C3 circuit," cannot be reproduced by this model (Explanation Part E).

<!-- ここでの min-entropy は測定分布からの値であり、量子的な乱数の安全性を保証する値ではありません。 -->

The min-entropy here is a value derived from the measurement distribution and does not guarantee the security of the quantum random numbers.

<!-- **結果と raw data の保存** -->

**Saving the results and the raw data**

<!-- 1. で `SAVE_SWEEP = True` にした場合、`run_noise_sweep` は測定時に各条件・試行の多数決で得たビット列（`.txt`）、5 ビットの文字列（`.shots.txt`）、条件・集計結果（`.txt.json`）を `RUN_DIR/5.9_noise_sweep/<ノイズの種類>/` に保存し、すべての条件の測定が終わったあとに、`5.9_noise_sweep/` に次の CSV を保存します。 -->

If `SAVE_SWEEP = True` in 1., `run_noise_sweep` saves the bit string from the majority vote (`.txt`), the 5-bit strings (`.shots.txt`), and the conditions and aggregated results (`.txt.json`) for each condition and trial in `RUN_DIR/5.9_noise_sweep/<noise type>/` during measurement, and after all conditions have been measured, it saves the following CSV files in `5.9_noise_sweep/`.

<!--
- `trials.csv`：試行ごとの全結果。
- `summary.csv`：条件ごとの平均・標準偏差・密度行列による値。
-->

- `trials.csv`: all results for each trial.
- `summary.csv`: per-condition means, standard deviations, and density-matrix values.

<!--
- `SAVE_SWEEP = False`（既定）の場合は、raw data も CSV も保存しません。
- CSV は、同じ実行の raw data と同じフォルダにだけ保存します。CSV だけが新しい結果に書き換わって、raw data と対応しなくなることはありません。
- 同じ `RUN_DIR` のまま測定のセルをもう一度実行すると、保存先が空でないので、測定を始める前にエラーになります。もう一度保存するときは、1. のセルを実行し直してください。
- 測定が途中で止まった場合、そのフォルダには CSV が作られません。CSV がないフォルダは、途中までの raw data です。
- CSV の集計結果から測定順のビット列は復元できないため、raw data が必要なら測定前に `SAVE_SWEEP = True` にしてください。
-->

- With `SAVE_SWEEP = False` (the default), neither the raw data nor the CSV files are saved.
- The CSV files are saved only in the same folder as the raw data of the same run. The CSV files are never replaced by newer results that no longer match the raw data.
- If you run the measurement cell again with the same `RUN_DIR`, the destination is not empty, so an error is raised before any measurement. To save again, rerun the cell in 1.
- If the measurement stops partway, no CSV files are created in that folder. A folder without CSV files holds the raw data of an incomplete run.
- The bit strings in measurement order cannot be recovered from the aggregated results in the CSV, so if you need raw data, set `SAVE_SWEEP = True` before measuring.

<!-- 下のセルは、保存した CSV の場所を表示します。 -->

The cell below shows where the CSV files were saved.

In [ ]:
# The CSV files are saved by run_noise_sweep in the cell above, in SWEEP_RAW_DIR together with the raw data.
if SWEEP_RAW_DIR is None:
    print("SAVE_SWEEP = False, so neither the raw data nor the CSV files were saved.")
else:
    for filename in ("trials.csv", "summary.csv"):
        path = Path(SWEEP_RAW_DIR) / filename
        print(("Saved: " if path.is_file() else "Not saved: ") + str(path))